# Demo · Captura con PLN — Enterprise Brain Glaciares (G8)
**Del PDF al conocimiento con fuente:** un documento en vivo + resultados del corpus completo (398 documentos).

### Guion (≈3 min)
| Celda | Qué se muestra | Qué decir |
|---|---|---|
| ⚙️ Preparación | — | *(ejecutar ANTES de la clase; tarda ~1 min)* |
| 1 | Un PDF del INAIGEM → texto con secciones | "Tomamos un informe real de 30 páginas; quitamos índice, cabeceras y detectamos sus secciones." |
| 2 | Ese texto → tripletas con evidencia | "Cada hecho dice de qué documento, página y oración salió: así el agente podrá citar." |
| 3 | Cifras del corpus completo | "Lo mismo, aplicado a 398 documentos de INAIGEM, ANA y SENAMHI, con 6 técnicas combinadas." |
| 4 | Preguntas de competencia | "Ya responde preguntas de varios saltos: laguna → cordillera → departamento." |
| 5 | Grafo de Palcacocha | "Así se verá el grafo de conocimiento la próxima semana en Neo4j." |

## ⚙️ Preparación — ejecutar antes de la clase

In [ ]:
import os, sys, json, time
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/proyecto_gc"
except ImportError:
    BASE = r"G:\Mi unidad\proyecto_gc"
CORPUS, TEXTO = os.path.join(BASE, "corpus_glaciares"), os.path.join(BASE, "texto_glaciares")
TRIP = os.path.join(BASE, "tripletas_glaciares_v2")
if not os.path.exists(os.path.join(TRIP, "tripletas_consolidadas.csv")):
    TRIP = os.path.join(BASE, "tripletas_glaciares")      # respaldo: resultados de la v1
DOC = "20.500.12748/394"                                   # Informe de inspección A06, laguna Islapata (INAIGEM, 2021)
os.system("pip install -q pymupdf")

In [ ]:
# Código de las personas 2 y 3 (no hace falta mostrarlo)
# -*- coding: utf-8 -*-
"""
Persona 2 — Del PDF al texto estructurado (caso Glaciares)
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG)

Entrada (Persona 1): papers.csv + pdfs/ + pdfs_manifest.csv  (398 documentos INAIGEM, ANA, SENAMHI)
Para cada PDF:
  1. Lee cada línea con su tipografía y posición (PyMuPDF).
  2. Detecta páginas escaneadas y, si Tesseract está instalado, les aplica OCR (español).
  3. Limpia: cabeceras y pies repetidos, números de página, índice / tabla de contenido,
     ligaduras y palabras cortadas con guion.
  4. Detecta secciones de informes técnicos en español (y de artículos en inglés) y las normaliza:
     resumen, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
     discusion, conclusiones, recomendaciones, referencias, anexos...
  5. Separa la primera página (institución, fecha), leyendas (Figura, Foto, Cuadro, Mapa) y tablas.
  6. Divide el texto en fragmentos con procedencia (documento, sección, páginas) y marca
     "pistas" del dominio (laguna, glaciar, peligro, poblado, obra, ubicación, medición)
     para que la Persona 3 priorice la extracción de entidades y relaciones.
Es reanudable: no reprocesa los PDFs que no cambiaron.
"""
import os, re, json, gzip, hashlib, unicodedata
from collections import Counter
from datetime import datetime, timezone

import pandas as pd

try:
    import pymupdf
except ImportError:  # versiones antiguas
    import fitz as pymupdf

VERSION = "2.0-glaciares"


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn")


def normalizar_texto(t):
    t = unicodedata.normalize("NFKC", t or "")  # ligaduras ﬁ -> fi, etc.
    t = t.replace("­", "").replace("​", "").replace("﻿", "")
    return re.sub(r"[ \t]+", " ", t).strip()


def nombre_archivo(paper_id):
    """Mismo nombre que usa la Persona 1 para el PDF (válido en Windows)."""
    return re.sub(r"[^\w.-]", "_", str(paper_id))


# ======================================================================
# 1. OCR (opcional, requiere Tesseract instalado)
# ======================================================================
def preparar_ocr(usar_ocr=True, ruta_tesseract=""):
    """Devuelve (disponible, idioma, mensaje)."""
    if not usar_ocr:
        return False, "", "OCR desactivado en la configuración"
    if ruta_tesseract:
        os.environ["TESSDATA_PREFIX"] = ruta_tesseract
    try:
        tessdata = os.environ.get("TESSDATA_PREFIX") or pymupdf.get_tessdata()
    except Exception:
        tessdata = ""
    if not tessdata or not os.path.isdir(tessdata):
        return False, "", ("Tesseract no está instalado: los PDFs escaneados usarán el resumen de los metadatos. "
                           "Ver la sección 0 para instalarlo.")
    os.environ["TESSDATA_PREFIX"] = tessdata
    idiomas = {f[:-12] for f in os.listdir(tessdata) if f.endswith(".traineddata")}
    if "spa" in idiomas:
        return True, "spa+eng" if "eng" in idiomas else "spa", f"OCR disponible (español) en {tessdata}"
    if "eng" in idiomas:
        return True, "eng", f"OCR disponible, pero sin el idioma español (spa.traineddata) en {tessdata}: se usará inglés"
    return False, "", f"Tesseract sin idiomas en {tessdata}"


# ======================================================================
# 2. Lectura de líneas con tipografía
# ======================================================================
def es_negrita(span):
    fuente = (span.get("font") or "").lower()
    return bool(span.get("flags", 0) & 16) or any(k in fuente for k in ("bold", "black", "heavy", "semibold", "negrita", ",b", "-b"))


def unir_spans(spans):
    """Une los tramos de una línea; si hay separación visual sin espacio, agrega uno."""
    out, x_prev = "", None
    for sp in spans:
        t = sp.get("text", "")
        if not t:
            continue
        x0 = sp["bbox"][0]
        if out and x_prev is not None and not out.endswith(" ") and not t.startswith(" ") \
                and x0 - x_prev > 0.15 * sp.get("size", 10):
            out += " "
        out += t
        x_prev = sp["bbox"][2]
    return out


def lineas_de_pagina(page, pno, textpage=None):
    lineas = []
    W, H = page.rect.width, page.rect.height
    d = page.get_text("dict", flags=pymupdf.TEXT_MEDIABOX_CLIP, textpage=textpage)
    for bi, b in enumerate(d.get("blocks", [])):
        if b.get("type") != 0:
            continue
        for li, ln in enumerate(b.get("lines", [])):
            dx, dy = ln.get("dir", (1, 0))
            if abs(dy) > 0.5:  # texto vertical en el margen
                continue
            spans = [s for s in ln.get("spans", []) if s.get("text", "").strip()]
            if not spans:
                continue
            texto = normalizar_texto(unir_spans(ln["spans"]))
            if not texto:
                continue
            principal = max(spans, key=lambda s: len(s["text"].strip()))
            x0, y0, x1, y1 = ln["bbox"]
            lineas.append({
                "page": pno, "block": bi, "line": li, "text": texto,
                "size": round(principal.get("size", 0), 1),
                "bold": sum(len(s["text"]) for s in spans if es_negrita(s)) >= 0.6 * sum(len(s["text"]) for s in spans),
                "x0": x0, "y0": y0, "x1": x1, "y1": y1, "W": W, "H": H,
            })
    return lineas


def cobertura_imagen(page):
    """Fracción de la página cubierta por imágenes (0 a 1)."""
    area = abs(page.rect)
    if not area:
        return 0.0
    total = 0.0
    try:
        for im in page.get_image_info():
            r = pymupdf.Rect(im["bbox"]) & page.rect
            total += abs(r)
    except Exception:
        return 0.0
    return min(1.0, total / area)


def leer_documento(doc, ocr=(False, ""), max_paginas_ocr=40, min_car_pagina=80, forzar_ocr=False):
    """Lee todas las páginas. Una página se trata como escaneada si casi no tiene texto y tiene
    imágenes, o si una imagen cubre la mayor parte de la hoja y el texto es poco (p. ej. un sello
    o una marca del repositorio sobre la foto de la página). Con OCR disponible se lee con Tesseract.
    forzar_ocr=True: se aplica OCR a toda página con imagen grande (para reparar documentos)."""
    ocr_ok, idioma = ocr
    lineas, escaneadas, con_ocr, errores_ocr = [], [], [], 0
    for pno, page in enumerate(doc):
        lp = lineas_de_pagina(page, pno)
        n_car = sum(len(l["text"]) for l in lp)
        cob = cobertura_imagen(page) if (n_car < 1500 or forzar_ocr) else 0.0
        escaneada = ((n_car < min_car_pagina and cob > 0) or (cob >= 0.6 and n_car < 600)
                     or (forzar_ocr and cob >= 0.3 and n_car < 1500))
        if escaneada:
            escaneadas.append(pno + 1)
            if ocr_ok and len(con_ocr) < max_paginas_ocr:
                try:
                    tp = page.get_textpage_ocr(language=idioma, dpi=200, full=True)
                    lp_ocr = lineas_de_pagina(page, pno, textpage=tp)
                    if sum(len(l["text"]) for l in lp_ocr) > n_car:
                        for l in lp_ocr:
                            l["ocr"] = True
                        lp = lp_ocr
                        con_ocr.append(pno + 1)
                except Exception:
                    errores_ocr += 1
        lineas += lp
    return lineas, {"paginas_escaneadas": escaneadas, "paginas_ocr": con_ocr, "errores_ocr": errores_ocr}


def quitar_cabeceras_pies(lineas, n_paginas, margen=0.09, min_frac=0.35):
    """Elimina líneas en los márgenes que se repiten en muchas páginas
    (p. ej. 'INAIGEM - Dirección de Investigación en Glaciares') y números de página."""
    def clave(l):
        return re.sub(r"\d+", "#", sin_tildes(l["text"].lower()))

    def en_margen(l):
        return l["y1"] < l["H"] * margen or l["y0"] > l["H"] * (1 - margen)

    conteo = {}
    for l in lineas:
        if en_margen(l):
            conteo.setdefault(clave(l), set()).add(l["page"])
    umbral = max(2, int(round(n_paginas * min_frac)))
    repetidas = {k for k, pags in conteo.items() if len(pags) >= umbral}
    salida, quitadas = [], 0
    for l in lineas:
        if en_margen(l) and (clave(l) in repetidas or
                             re.fullmatch(r"(p[aá]g(ina)?\.?\s*)?[\divxlc\-–— /de]{1,12}", l["text"].lower())):
            quitadas += 1
            continue
        salida.append(l)
    return salida, quitadas


# ======================================================================
# 3. Índice (tabla de contenido)
# ======================================================================
PAT_LINEA_INDICE = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ].*?(\.{3,}|…{2,}|(\s\.){3,}|_{3,})\s*\d{1,3}\s*$")
PAT_FIN_NUMERO = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ)]\s+\d{1,3}\s*$")


def paginas_indice(lineas, max_pagina=10):
    """Páginas (entre las primeras) donde la mayoría de líneas terminan en número de página."""
    por_pag = {}
    for l in lineas:
        if l["page"] < max_pagina:
            por_pag.setdefault(l["page"], []).append(l["text"])
    out = set()
    for p, textos in por_pag.items():
        n = sum(bool(PAT_LINEA_INDICE.search(t) or PAT_FIN_NUMERO.search(t)) for t in textos)
        cabecera = any(re.match(r"^(indice|contenido|tabla de contenido|indice general)\b", sin_tildes(t.lower()))
                       for t in textos[:6])
        if n >= 5 and (n >= 0.4 * len(textos) or cabecera):
            out.add(p)
    return out


# ======================================================================
# 4. Encabezados y tipos de sección (español + inglés)
# ======================================================================
PAT_NUMERADO = re.compile(
    r"^(?P<num>\d{1,2}(?:\.\d{1,2}){0,3}|[A-H](?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)\-–]?\s+"
    r"(?P<titulo>[A-ZÁÉÍÓÚÑÜ¿(][^\n]{1,110})$")
PAT_SOLO_NUMERO = re.compile(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)]?$")
PAT_CAPITULO = re.compile(r"^cap[ií]tulo\s+(?P<num>[ivxlc\d]+)\b\.?\s*[:\-–.]?\s*(?P<titulo>.*)$", re.I)
PAT_LEYENDA = re.compile(
    r"^(figura|fig\.|foto(graf[ií]a)?|imagen|tabla|cuadro|gr[aá]fico|mapa|l[aá]mina|plano|esquema|"
    r"figure|table|photo)\s*(n\s*[°º.]\s*|nro\.?\s*|no\.?\s*)?\d+[a-z]?(\.\d+)?\s*[:.\-–|]", re.I)

NOMBRES_CONOCIDOS = {
    # español
    "resumen", "resumen ejecutivo", "presentacion", "prologo", "introduccion", "objetivo", "objetivos",
    "objetivo general", "objetivos especificos", "antecedentes", "marco teorico", "marco conceptual",
    "justificacion", "alcance", "alcances", "ubicacion", "ubicacion y acceso", "accesibilidad",
    "area de estudio", "zona de estudio", "ambito de estudio", "descripcion del area de estudio",
    "metodologia", "materiales y metodos", "metodos", "trabajos de campo", "trabajo de campo",
    "resultados", "resultados y discusion", "discusion", "analisis", "evaluacion", "diagnostico",
    "conclusiones", "conclusion", "conclusiones y recomendaciones", "recomendaciones",
    "referencias", "referencias bibliograficas", "bibliografia", "literatura citada",
    "anexos", "anexo", "apendice", "panel fotografico", "registro fotografico", "agradecimientos",
    "indice", "contenido", "tabla de contenido", "glosario", "abreviaturas", "siglas y acronimos",
    # inglés (artículos científicos del corpus)
    "abstract", "summary", "introduction", "background", "related work", "study area", "data and methods",
    "methods", "methodology", "materials and methods", "data", "results", "discussion", "conclusion",
    "conclusions", "acknowledgments", "acknowledgements", "references", "appendix", "supplementary material",
}

TIPOS_REGEX = [
    ("indice", r"^(indice|contenido|tabla de contenido|indice general|lista de (figuras|tablas|cuadros|fotos|fotografias|mapas|anexos)|"
               r"indice de (figuras|tablas|cuadros|fotografias|mapas|anexos)|glosario|abreviaturas|siglas)"),
    ("agradecimientos", r"agradecimiento|acknowledg"),
    ("referencias", r"^(referencias|bibliografia|literatura citada|fuentes consultadas|references|bibliography)"),
    ("anexos", r"^(anexo|apendice|appendi|panel fotografico|registro fotografico|supplementa)"),
    ("resumen", r"^(resumen|resumen ejecutivo|abstract|summary|sumario)$"),
    ("presentacion", r"^(presentacion|prologo|prefacio|foreword|preface)"),
    ("introduccion", r"^(introduc|generalidades)"),
    ("objetivos", r"objetivo|^finalidad|^alcances?$|^justificacion"),
    ("antecedentes", r"antecedente|marco teorico|marco conceptual|estado del arte|background|related work|literature"),
    ("area_estudio", r"ubicacion|area de estudio|zona de estudio|ambito|accesibilidad|localizacion|study area|"
                     r"descripcion (general )?(de la|del) (zona|area|laguna|glaciar|cuenca|subcuenca|nevado)|"
                     r"caracteristicas (generales|fisicas|de la laguna|del glaciar|de la cuenca)|geografia|clima de la zona"),
    ("conclusiones", r"^conclusi|conclusion"),
    ("recomendaciones", r"recomendaci"),
    ("discusion", r"^discusi|discussion"),
    ("metodologia", r"metodolog|metodo|materiales y|procedimiento|trabajos? de campo|methods?\b|methodology|"
                    r"^data\b|datos y|levantamiento|instrumentacion|equipos? utilizados"),
    ("resultados", r"resultado|analisis|evaluacion|diagnostico|caracterizacion|inspeccion|monitoreo|"
                   r"inventario|batimetri|peligro|riesgo|results|evaluation|analysis"),
]
TIPOS_SIN_FRAGMENTOS = ("indice", "referencias", "agradecimientos", "front_matter")


def tipo_de_titulo(titulo):
    t = re.sub(r"[^a-z ]", " ", sin_tildes(titulo.lower()))
    t = re.sub(r"\s+", " ", t).strip()
    for tipo, pat in TIPOS_REGEX:
        if re.search(pat, t):
            return tipo
    return "otro"


def limpiar_titulo(t):
    return re.sub(r"\s+", " ", t).strip(" .:-–")


def detectar_encabezado(l, cuerpo):
    """Devuelve (numero, titulo, nivel) si la línea es un encabezado; si no, None."""
    texto = l["text"].strip()
    if len(texto) > 120 or len(texto.split()) > 16 or PAT_LEYENDA.match(texto):
        return None
    letras = re.sub(r"[^A-Za-zÁÉÍÓÚÑÜáéíóúñü]", "", texto)
    mayusculas = len(letras) >= 4 and letras.isupper()
    destacado = l["bold"] or l["size"] >= cuerpo + 0.8 or mayusculas
    m = PAT_CAPITULO.match(texto)
    if m and (destacado or len(texto.split()) <= 6):
        return m.group("num").upper(), limpiar_titulo(m.group("titulo")) or texto, 1
    sin_num = limpiar_titulo(re.sub(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[A-H]|[IVX]{1,5})[.)\-–]?\s+", "", texto))
    if sin_tildes(sin_num.lower()) in NOMBRES_CONOCIDOS and (destacado or l["size"] >= cuerpo - 0.2):
        num = texto[: len(texto) - len(sin_num)].strip(" .)-–") if texto.lower() != sin_num.lower() else ""
        return num, sin_num, (num.count(".") + 1) if num else 1
    m = PAT_NUMERADO.match(texto)
    if m and destacado:
        num, titulo = m.group("num"), limpiar_titulo(m.group("titulo"))
        if re.match(r"^[A-H]$", num) and not (l["bold"] or mayusculas):
            return None
        if titulo.endswith(".") and len(titulo.split()) > 7:
            return None  # oración numerada (lista de conclusiones o recomendaciones)
        if sum(c.isdigit() for c in titulo) > 6:
            return None
        nivel = num.count(".") + 1 if num[0].isdigit() else 1
        return num, titulo, nivel
    return None


# ======================================================================
# 5. Procesamiento de un PDF
# ======================================================================
PAT_INSTITUCION = re.compile(
    r"\b(INAIGEM|ANA|SENAMHI|INDECI|CENEPRED|INGEMMET|IGP|INRENA|ELECTROPER[UÚ]|HIDRANDINA|EGENOR|"
    r"autoridad nacional del agua|instituto\s+\w+|servicio nacional|ministerio|direcci[oó]n|subdirecci[oó]n|"
    r"unidad de glaciolog\w*|gobierno regional|municipalidad|universi\w+|facultad|"
    r"university|institute|department|laborator\w+|centro de)\b", re.I)
PAT_EMAIL = re.compile(r"[\w.+-]+@[\w-]+(\.[\w-]+)+")
PAT_FECHA = re.compile(r"\b(enero|febrero|marzo|abril|mayo|junio|julio|agosto|se(p)?tiembre|octubre|noviembre|diciembre)"
                       r"\s*(de|del)?\s*(19|20)\d{2}\b", re.I)

PISTAS = {
    "laguna": r"\blagunas?\b|\blake\b|\blakes\b",
    "glaciar": r"\bglaciar\w*|\bnevados?\b|\bglacier\w*|\bcobertura glaciar|\bmasa de hielo",
    "peligro": r"riesgo|peligr\w*|aluvi[oó]n\w*|desborde|glof|avalanch\w*|huaico|huayco|deslizamiento|vulnerab\w*|hazard|outburst",
    "poblado": r"centros? poblados?|ciudad de|distrito de|localidad|caser[ií]o|comunidad campesina|poblaci[oó]n|habitantes|viviendas",
    "obra": r"obras? de (seguridad|control)|\bdique\w*|t[uú]nel\w*|canal de (desag[uü]e|descarga)|sif[oó]n\w*|aliviadero|"
            r"represamiento|presa de|desembalse|sistema de alerta temprana",
    "ubicacion": r"cordillera|subcuenca|microcuenca|cuenca del?|departamento|provincia de|regi[oó]n (de )?[A-ZÁÉÍÓÚ]|quebrada",
    "medicion": r"\d+(?:[.,]\d+)?\s?(?:m\.?\s?s\.?\s?n\.?\s?m\.?|msnm|km2|km²|m2|m²|m3|m³|hm3|hm³|mmc|ha\b|%|m/a|m/año)",
}
PISTAS_RE = {k: re.compile(v, re.I) for k, v in PISTAS.items()}
STOP_ES = set("de la que el en los del se las por un para con una su al lo como más pero sus le ya o fue este ha sí porque esta son entre "
              "cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso".split())
STOP_EN = set("the of and to in is that for it as with was on be by this are from at or an which have has were not but their these".split())


def idioma(texto):
    pal = re.findall(r"[a-záéíóúñ]+", texto[:20000].lower())
    es, en = sum(p in STOP_ES for p in pal), sum(p in STOP_EN for p in pal)
    if es + en < 20:
        return "desconocido"
    return "es" if es >= en else "en"


def unir_lineas(textos):
    """Une líneas de un párrafo reparando palabras cortadas con guion."""
    out = ""
    for t in textos:
        if not out:
            out = t
        elif out.endswith("-") and t[:1].islower() and not re.search(r"-\w+-$", out):
            out = out[:-1] + t
        else:
            out += " " + t
    return out.strip()


def procesar_pdf(ruta, paper_id, titulo_meta="", abstract_meta="", extraer_tablas=True,
                 max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40, forzar_ocr=False):
    res = {"paper_id": paper_id, "version": VERSION, "title": titulo_meta, "flags": [], "procesado": ahora()}
    try:
        doc = pymupdf.open(ruta)
    except Exception as e:
        res["flags"].append("pdf_no_abre")
        res["error"] = str(e)[:300]
        return res
    res["n_pages"] = doc.page_count

    lineas, info = leer_documento(doc, ocr, max_paginas_ocr, forzar_ocr=forzar_ocr)
    res.update(info)
    res["ocr_aplicado"] = bool(info["paginas_ocr"])
    res["ocr_disponible"] = bool(ocr[0])
    n_esc = len(info["paginas_escaneadas"])
    if n_esc:
        res["flags"].append(f"escaneado ({n_esc}/{doc.page_count} pág.)" if n_esc >= 0.5 * doc.page_count
                            else f"paginas_escaneadas ({n_esc})")
    if n_esc and not info["paginas_ocr"]:
        res["flags"].append("sin_ocr")

    # Tablas (antes del texto para no mezclar sus celdas con los párrafos). No en páginas OCR.
    tablas, zonas_tabla = [], {}
    if extraer_tablas:
        for pno in range(min(doc.page_count, max_paginas_tablas)):
            if pno + 1 in info["paginas_escaneadas"]:
                continue
            try:
                for t in doc[pno].find_tables().tables:
                    md = t.to_markdown(clean=True) if hasattr(t, "to_markdown") else ""
                    if md and md.count("\n") >= 2:
                        tablas.append({"page": pno + 1, "bbox": [round(v, 1) for v in t.bbox], "markdown": md})
                        zonas_tabla.setdefault(pno, []).append(t.bbox)
            except Exception:
                pass

    texto_total = " ".join(l["text"] for l in lineas)
    res["pdf_text_chars"] = len(texto_total)
    if texto_total.count("(cid:") > 20 or texto_total.count("�") > 50:
        res["flags"].append("texto_con_errores_de_codificacion")
    res["idioma"] = idioma(texto_total)

    lineas, n_quitadas = quitar_cabeceras_pies(lineas, doc.page_count)
    res["lineas_cabecera_pie_quitadas"] = n_quitadas
    pags_indice = paginas_indice(lineas)
    res["paginas_indice"] = sorted(p + 1 for p in pags_indice)

    def en_tabla(l):
        for (x0, y0, x1, y1) in zonas_tabla.get(l["page"], []):
            if l["x0"] >= x0 - 2 and l["x1"] <= x1 + 2 and l["y0"] >= y0 - 2 and l["y1"] <= y1 + 2:
                return True
        return False

    tamanos = Counter()
    for l in lineas:
        tamanos[l["size"]] += len(l["text"])
    cuerpo = tamanos.most_common(1)[0][0] if tamanos else 10.0
    res["font_size_cuerpo"] = cuerpo

    # Título: línea(s) con la mayor tipografía en la primera página (portada)
    p1 = [l for l in lineas if l["page"] == 0]
    titulo_ids = set()
    if p1:
        mayor = max(l["size"] for l in p1)
        if not titulo_meta:
            res["title"] = " ".join(l["text"] for l in p1 if l["size"] == mayor)[:300]
        if mayor >= cuerpo + 2:
            titulo_ids = {(l["page"], l["block"], l["line"]) for l in p1 if l["size"] == mayor}

    secciones = [{"heading": "PORTADA", "num": "", "level": 0, "type": "front_matter", "parrafos": []}]
    leyendas, lineas_front, n_lineas_indice = [], [], 0
    buffer, buf_key, buf_pag = [], None, None
    pendiente_num = None

    def cerrar_parrafo():
        nonlocal buffer, buf_key, buf_pag
        if buffer:
            texto = unir_lineas(buffer)
            if PAT_LEYENDA.match(texto) and len(texto) < 400:
                leyendas.append({"page": buf_pag + 1, "text": texto, "section": secciones[-1]["heading"]})
            elif len(texto) > 1:
                previos = secciones[-1]["parrafos"]
                if previos and previos[-1]["text"].endswith("-") and texto[:1].islower():
                    previos[-1]["text"] = previos[-1]["text"][:-1] + texto   # palabra cortada entre bloques
                else:
                    previos.append({"page": buf_pag + 1, "text": texto})
        buffer, buf_key, buf_pag = [], None, None

    for l in lineas:
        if (l["page"], l["block"], l["line"]) in titulo_ids or en_tabla(l):
            continue
        if l["page"] in pags_indice or PAT_LINEA_INDICE.search(l["text"]):
            n_lineas_indice += 1
            continue
        if pendiente_num is not None:
            l = dict(l, text=f"{pendiente_num} {l['text']}")
            pendiente_num = None
        if PAT_SOLO_NUMERO.match(l["text"]) and (l["bold"] or l["size"] >= cuerpo + 0.8):
            cerrar_parrafo()
            pendiente_num = l["text"].strip(" .)")
            continue
        enc = detectar_encabezado(l, cuerpo)
        if enc:
            cerrar_parrafo()
            num, titulo, nivel = enc
            secciones.append({"heading": titulo, "num": num, "level": nivel, "type": tipo_de_titulo(titulo),
                              "page_start": l["page"] + 1, "parrafos": []})
            continue
        if len(secciones) == 1:
            lineas_front.append(l["text"])
        clave = (l["page"], l["block"])
        if clave != buf_key:
            cerrar_parrafo()
            buf_key, buf_pag = clave, l["page"]
        buffer.append(l["text"])
    cerrar_parrafo()
    res["lineas_indice_quitadas"] = n_lineas_indice

    # Tipos: herencia en subsecciones y deducción por posición
    padre = None
    for s in secciones[1:]:
        if s["level"] <= 1:
            padre = s
        elif s["type"] == "otro" and padre is not None:
            s["type"] = padre["type"]
    i_intro = next((i for i, s in enumerate(secciones) if s["type"] in ("introduccion", "objetivos", "antecedentes")), None)
    i_fin = next((i for i, s in enumerate(secciones) if s["type"] in ("conclusiones", "recomendaciones", "referencias")), None)
    i_ref = next((i for i, s in enumerate(secciones) if s["type"] == "referencias"), None)
    for i, s in enumerate(secciones):
        if s["type"] != "otro":
            continue
        if i_ref is not None and i > i_ref:
            s["type"] = "anexos"
        elif i_intro is not None and i > i_intro and (i_fin is None or i < i_fin):
            s["type"] = "desarrollo"   # capítulo propio del informe (p. ej. "Descripción de peligros")
    if i_ref is not None:
        for s in secciones[i_ref + 1:]:
            if s["type"] not in ("agradecimientos", "referencias", "anexos", "conclusiones", "recomendaciones"):
                s["tipo_original"] = s["type"]
                s["type"] = "anexos"

    # Resumen: sección del PDF o, si no hay, el de los metadatos (lo normal en informes técnicos)
    resumen_sec = next((s for s in secciones if s["type"] == "resumen"), None)
    resumen = " ".join(p["text"] for p in resumen_sec["parrafos"]) if resumen_sec else (abstract_meta or "")
    res["resumen_fuente"] = "pdf" if resumen_sec else ("metadatos" if abstract_meta else "no_encontrado")

    front_texto = "\n".join(p["text"] for p in secciones[0]["parrafos"])
    p1_texto = "\n".join(l["text"] for l in lineas if l["page"] == 0)
    candidatas = []
    for ln in lineas_front[:60] + p1_texto.split("\n")[:60]:
        if PAT_INSTITUCION.search(ln) and not PAT_EMAIL.search(ln) and 3 < len(ln) < 200:
            candidatas.append(ln.strip(" ,"))
    fechas = [m.group(0) for m in PAT_FECHA.finditer(front_texto + "\n" + p1_texto)]
    res.update({
        "resumen": resumen, "front_matter": front_texto[:5000],
        "instituciones_candidatas": list(dict.fromkeys(candidatas))[:20],
        "fechas_portada": list(dict.fromkeys(fechas))[:5],
        "emails": list(dict.fromkeys(m.group(0) for m in PAT_EMAIL.finditer(front_texto)))[:20],
        "first_page_text": p1_texto[:6000],
        "captions": leyendas, "tables": tablas,
    })

    salida = []
    for s in secciones:
        texto = "\n\n".join(p["text"] for p in s["parrafos"])
        if s["type"] == "front_matter" and not texto:
            continue
        paginas = [p["page"] for p in s["parrafos"]] or [s.get("page_start", 1)]
        salida.append({"idx": len(salida), "heading": s["heading"], "num": s["num"], "level": s["level"],
                       "type": s["type"], "page_start": min(paginas), "page_end": max(paginas),
                       "n_chars": len(texto), "parrafos": s["parrafos"], "text": texto})
    res["sections"] = salida

    tipos = {s["type"] for s in salida}
    principales = [s for s in salida if s["level"] == 1 and s["type"] != "front_matter"]
    sin_estructura = len(principales) < 3
    excluir = ("indice", "referencias", "agradecimientos") if sin_estructura else TIPOS_SIN_FRAGMENTOS
    n_util = sum(s["n_chars"] for s in salida if s["type"] not in excluir)
    if n_util < 1500:
        res["flags"].append("sin_texto_util")
    elif sin_estructura:
        res["flags"].append("estructura_no_detectada")
    res["tipos_detectados"] = sorted(tipos)
    res["n_sections"] = len(salida)
    res["n_chars_texto"] = sum(s["n_chars"] for s in salida)
    res["n_chars_util"] = n_util
    doc.close()
    return res


# ======================================================================
# 6. Fragmentación con procedencia y pistas del dominio
# ======================================================================
def dividir_oraciones(texto):
    return [o for o in re.split(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÑ¿¡(\[\d])", texto) if o.strip()]


def pistas(texto):
    return [k for k, p in PISTAS_RE.items() if p.search(texto)]


def fragmentar(res, meta=None, objetivo=1500, maximo=2200, solape=200):
    """Fragmentos por sección respetando párrafos. Sin estructura: todo el cuerpo.
    Sin texto útil (escaneado sin OCR): un único fragmento con título + resumen de los metadatos."""
    meta = meta or {}
    pid = res["paper_id"]
    base = {"paper_id": pid, "fuente": meta.get("fuente", ""), "title": meta.get("title", res.get("title", "")),
            "year": meta.get("year", "")}
    chunks = []
    secciones = res.get("sections", [])
    flags = res.get("flags", [])
    if "sin_texto_util" in flags or not secciones:
        texto = (base["title"] + ". " + (res.get("resumen") or meta.get("abstract", "") or "")).strip()
        if len(texto) > 20:
            p = pistas(texto)
            chunks.append({**base, "chunk_id": f"{pid}::meta::c000", "section_idx": -1, "section_type": "resumen",
                           "heading": "Resumen (metadatos)", "page_start": None, "page_end": None,
                           "text": texto, "n_chars": len(texto), "pistas": p, "prioridad": len(p),
                           "fuente_texto": "metadatos"})
        return chunks
    if "estructura_no_detectada" in flags:
        todos = [p for s in secciones for p in s["parrafos"] if s["type"] not in ("indice", "referencias", "agradecimientos")]
        secciones = [{"idx": 0, "heading": "CUERPO", "type": "cuerpo", "parrafos": todos}]
    paginas_ocr = set(res.get("paginas_ocr", []))
    for s in secciones:
        if s["type"] in TIPOS_SIN_FRAGMENTOS:
            continue
        piezas = []
        for p in s["parrafos"]:
            if len(p["text"]) <= maximo:
                piezas.append(p)
            else:
                actual = ""
                for o in dividir_oraciones(p["text"]):
                    while len(o) > maximo:   # oración enorme (texto OCR sin puntos)
                        piezas.append({"page": p["page"], "text": o[:objetivo]})
                        o = o[objetivo:]
                    if actual and len(actual) + len(o) + 1 > objetivo:
                        piezas.append({"page": p["page"], "text": actual})
                        actual = o
                    else:
                        actual = (actual + " " + o).strip()
                if actual:
                    piezas.append({"page": p["page"], "text": actual})
        texto, paginas, k = "", [], 0

        def emitir():
            nonlocal k
            t = texto.strip()
            if len(t) >= 40:
                p = pistas(t)
                chunks.append({
                    **base, "chunk_id": f"{pid}::s{s['idx']:02d}::c{k:03d}", "section_idx": s["idx"],
                    "section_type": s["type"], "heading": s["heading"],
                    "page_start": min(paginas), "page_end": max(paginas), "text": t, "n_chars": len(t),
                    "pistas": p, "prioridad": len([x for x in p if x != "medicion"]) + ("medicion" in p) * 0.5,
                    "fuente_texto": "ocr" if set(paginas) & paginas_ocr else "pdf",
                })
                k += 1

        for p in piezas:
            if texto and len(texto) + len(p["text"]) + 2 > objetivo:
                emitir()
                arrastre = ""
                for o in reversed(dividir_oraciones(texto)):
                    if len(arrastre) + len(o) > solape:
                        break
                    arrastre = (o + " " + arrastre).strip()
                texto, paginas = arrastre, paginas[-1:]
            texto = (texto + "\n\n" + p["text"]).strip()
            paginas.append(p["page"])
        emitir()
    return chunks


# ======================================================================
# 7. Proceso por lotes (reanudable)
# ======================================================================
def ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1=None):
    if manifiesto_p1 is not None and pid in manifiesto_p1.index:
        archivo = manifiesto_p1.loc[pid, "archivo"]
        if isinstance(archivo, str) and archivo:
            return os.path.join(carpeta_pdfs, archivo)
    return os.path.join(carpeta_pdfs, nombre_archivo(pid) + ".pdf")


def fila_manifiesto(pid, res, sha, reutilizado=False):
    flags = res.get("flags", [])
    estado = ("error" if "pdf_no_abre" in flags else
              "solo_metadatos" if "sin_texto_util" in flags else
              "ok_ocr" if res.get("ocr_aplicado") else "ok")
    return {"paper_id": pid, "estado": estado, "sha256_pdf": sha, "version": VERSION,
            "n_pages": res.get("n_pages"), "idioma": res.get("idioma", ""),
            "paginas_escaneadas": len(res.get("paginas_escaneadas", [])),
            "paginas_ocr": len(res.get("paginas_ocr", [])),
            "n_sections": res.get("n_sections", 0), "n_chars_util": res.get("n_chars_util", 0),
            "resumen_fuente": res.get("resumen_fuente", ""),
            "n_tablas": len(res.get("tables", [])), "n_leyendas": len(res.get("captions", [])),
            "tipos_detectados": ";".join(res.get("tipos_detectados", [])),
            "flags": ";".join(flags), "procesado": res.get("procesado", ahora()),
            "reutilizado": reutilizado}


def procesar_corpus(papers, carpeta_pdfs, carpeta_salida, manifiesto_p1=None, extraer_tablas=True,
                    max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40, forzar=False, log=print):
    carpeta_json = os.path.join(carpeta_salida, "json")
    os.makedirs(carpeta_json, exist_ok=True)
    ruta_man = os.path.join(carpeta_salida, "texto_manifest.csv")
    previo = (pd.read_csv(ruta_man, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
              if os.path.exists(ruta_man) else None)
    filas, resultados = [], {}
    total = len(papers)
    for i, f in enumerate(papers.itertuples(), start=1):
        pid = str(f.paper_id)
        ruta_pdf = ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1)
        ruta_json = os.path.join(carpeta_json, nombre_archivo(pid) + ".json")
        if not os.path.exists(ruta_pdf):
            filas.append({"paper_id": pid, "estado": "sin_pdf", "flags": "sin_pdf", "procesado": ahora()})
            log(f"   ❌ {pid}: no está el PDF ({os.path.basename(ruta_pdf)})")
            continue
        sha = sha256_archivo(ruta_pdf)
        reutilizar = (not forzar and previo is not None and pid in previo.index and os.path.exists(ruta_json)
                      and previo.loc[pid, "sha256_pdf"] == sha and str(previo.loc[pid, "version"]) == VERSION)
        res = None
        if reutilizar:
            with open(ruta_json, encoding="utf-8") as fh:
                res = json.load(fh)
            # si antes no había OCR y ahora sí, se reprocesan los escaneados
            if ocr[0] and res.get("paginas_escaneadas") and not res.get("ocr_disponible"):
                reutilizar, res = False, None
        if res is None:
            res = procesar_pdf(ruta_pdf, pid, getattr(f, "title", "") or "", getattr(f, "abstract", "") or "",
                               extraer_tablas, max_paginas_tablas, ocr, max_paginas_ocr)
            res["pdf_sha256"] = sha
            with open(ruta_json, "w", encoding="utf-8") as fh:
                json.dump(res, fh, ensure_ascii=False)
        resultados[pid] = res
        filas.append(fila_manifiesto(pid, res, sha, reutilizar))
        if i % 10 == 0 or i == total:
            log(f"   {i} / {total} procesados")
            pd.DataFrame(filas).to_csv(ruta_man, index=False, encoding="utf-8")
    manifiesto = pd.DataFrame(filas)
    manifiesto.to_csv(ruta_man, index=False, encoding="utf-8")
    return resultados, manifiesto


def cargar_resultados(papers, carpeta_salida):
    """Lee los resultados ya guardados en json/ (no reprocesa nada)."""
    carpeta_json = os.path.join(carpeta_salida, "json")
    resultados = {}
    for pid in papers.paper_id.astype(str):
        ruta = os.path.join(carpeta_json, nombre_archivo(pid) + ".json")
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as fh:
                resultados[pid] = json.load(fh)
    return resultados


def diagnosticar_pdf(ruta, max_paginas=15):
    """Por qué un PDF quedó sin texto: caracteres e imágenes por página."""
    try:
        doc = pymupdf.open(ruta)
    except Exception as e:
        return {"error": str(e)[:200]}
    car, cob, muestra = [], [], ""
    for page in list(doc)[:max_paginas]:
        t = page.get_text()
        car.append(len(t.strip()))
        cob.append(cobertura_imagen(page))
        if not muestra and len(t.strip()) > 5:
            muestra = re.sub(r"\s+", " ", t.strip())[:120]
    n = doc.page_count
    doc.close()
    media_cob = sum(cob) / max(len(cob), 1)
    media_car = sum(car) / max(len(car), 1)
    if media_cob >= 0.5 and media_car < 1500:
        causa = "escaneado con poco texto (sello o marca) → OCR"
    elif media_cob < 0.05 and media_car < 80:
        causa = "sin texto ni imágenes (¿vectorial o dañado?)"
    elif media_cob < 0.5 and media_car < 300:
        causa = "mapa, plano o lámina con poco texto"
    else:
        causa = "revisar a mano"
    return {"paginas": n, "car_por_pagina": round(media_car), "cobertura_imagen": round(media_cob, 2),
            "texto_muestra": muestra, "causa_probable": causa}


def reprocesar_documentos(ids, papers, carpeta_pdfs, carpeta_salida, manifiesto_p1=None, ocr=(False, ""),
                          max_paginas_ocr=40, extraer_tablas=True, max_paginas_tablas=80, log=print):
    """Reprocesa solo los documentos indicados con OCR forzado y actualiza json/ y texto_manifest.csv."""
    meta = papers.set_index("paper_id")
    carpeta_json = os.path.join(carpeta_salida, "json")
    ruta_man = os.path.join(carpeta_salida, "texto_manifest.csv")
    man = pd.read_csv(ruta_man, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
    cambios = []
    for i, pid in enumerate(ids, start=1):
        ruta_pdf = ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1)
        if not os.path.exists(ruta_pdf):
            log(f"   ❌ {pid}: no está el PDF")
            continue
        antes = man.loc[pid, "estado"] if pid in man.index else ""
        m = meta.loc[pid] if pid in meta.index else {}
        titulo = m.get("title", "") if len(m) else ""
        resumen = m.get("abstract", "") if len(m) else ""
        res = procesar_pdf(ruta_pdf, pid, titulo if isinstance(titulo, str) else "",
                           resumen if isinstance(resumen, str) else "", extraer_tablas, max_paginas_tablas,
                           ocr, max_paginas_ocr, forzar_ocr=True)
        res["pdf_sha256"] = sha256_archivo(ruta_pdf)
        res["reparado"] = ahora()
        with open(os.path.join(carpeta_json, nombre_archivo(pid) + ".json"), "w", encoding="utf-8") as fh:
            json.dump(res, fh, ensure_ascii=False)
        fila = fila_manifiesto(pid, res, res["pdf_sha256"])
        for k, v in fila.items():
            if k != "paper_id":
                man.loc[pid, k] = v
        cambios.append({"paper_id": pid, "antes": antes, "ahora": fila["estado"],
                        "paginas_ocr": fila["paginas_ocr"], "car_utiles": fila["n_chars_util"]})
        log(f"   {i}/{len(ids)} {pid}: {antes} → {fila['estado']} ({fila['paginas_ocr']} pág. OCR, "
            f"{fila['n_chars_util']:,} car.)")
        man.reset_index().to_csv(ruta_man, index=False, encoding="utf-8")
    return pd.DataFrame(cambios), man.reset_index()


# ======================================================================
# 8. Salidas, reporte y documentación
# ======================================================================
def _jsonl(ruta, filas, comprimir=False):
    abrir = gzip.open if comprimir else open
    with abrir(ruta, "wt", encoding="utf-8") as fh:
        for x in filas:
            fh.write(json.dumps(x, ensure_ascii=False, default=str) + "\n")


def construir_salidas(resultados, papers, carpeta_salida, objetivo=1500, maximo=2200, solape=200):
    meta = papers.set_index("paper_id")
    carpeta_txt = os.path.join(carpeta_salida, "textos")
    os.makedirs(carpeta_txt, exist_ok=True)
    secciones, chunks, primeras, leyendas, tablas, docs = [], [], [], [], [], []
    for pid, r in resultados.items():
        m = meta.loc[pid].to_dict() if pid in meta.index else {}
        m = {k: ("" if (isinstance(v, float) and pd.isna(v)) else v) for k, v in m.items()}
        for s in r.get("sections", []):
            secciones.append({"paper_id": pid, "section_idx": s["idx"], "heading": s["heading"], "num": s["num"],
                              "level": s["level"], "section_type": s["type"], "page_start": s["page_start"],
                              "page_end": s["page_end"], "n_chars": s["n_chars"], "text": s["text"]})
        ch = fragmentar(r, m, objetivo, maximo, solape)
        chunks += ch
        # Texto limpio completo (sin índice ni referencias), para leer y para el RAG
        cuerpo = "\n\n".join(f"## {s['heading']}\n\n{s['text']}" for s in r.get("sections", [])
                             if s["type"] not in ("indice", "referencias") and s["text"])
        if not cuerpo:
            cuerpo = "\n\n".join(c["text"] for c in ch)
        with open(os.path.join(carpeta_txt, nombre_archivo(pid) + ".txt"), "w", encoding="utf-8") as fh:
            fh.write(f"# {m.get('title', r.get('title', ''))}\n\nFuente: {m.get('fuente', '')} · Año: {m.get('year', '')} · "
                     f"paper_id: {pid}\n\n{cuerpo}\n")
        primeras.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""),
                         "instituciones_candidatas": r.get("instituciones_candidatas", []),
                         "fechas_portada": r.get("fechas_portada", []), "emails": r.get("emails", []),
                         "first_page_text": r.get("first_page_text", "")})
        for c in r.get("captions", []):
            leyendas.append({"paper_id": pid, **c, "tipo": sin_tildes(c["text"].split()[0].lower().rstrip(".:"))})
        for k, t in enumerate(r.get("tables", [])):
            tablas.append({"table_id": f"{pid}::t{k:02d}", "paper_id": pid, **t})
        docs.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""), "year": m.get("year", ""),
                     "eje_principal": m.get("eje_principal", ""), "idioma": r.get("idioma", ""),
                     "n_pages": r.get("n_pages"), "n_chars_util": r.get("n_chars_util", 0),
                     "fuente_texto": ("metadatos" if ch and ch[0]["fuente_texto"] == "metadatos"
                                      else "pdf+ocr" if r.get("ocr_aplicado") else "pdf"),
                     "n_secciones": r.get("n_sections", 0), "n_fragmentos": len(ch),
                     "n_tablas": len(r.get("tables", [])), "resumen_fuente": r.get("resumen_fuente", ""),
                     "secciones": ";".join(r.get("tipos_detectados", [])), "observaciones": ";".join(r.get("flags", []))})
    df_sec = pd.DataFrame(secciones)
    df_chunks = pd.DataFrame(chunks)
    df_docs = pd.DataFrame(docs)
    df_sec.to_csv(os.path.join(carpeta_salida, "sections.csv"), index=False, encoding="utf-8")
    df_docs.to_csv(os.path.join(carpeta_salida, "documentos_procesados.csv"), index=False, encoding="utf-8")
    pd.DataFrame(leyendas).to_csv(os.path.join(carpeta_salida, "captions.csv"), index=False, encoding="utf-8")
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl"), chunks)
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl.gz"), chunks, comprimir=True)   # para GitHub (pesa menos)
    _jsonl(os.path.join(carpeta_salida, "first_pages.jsonl"), primeras)
    _jsonl(os.path.join(carpeta_salida, "tables.jsonl"), tablas)
    return df_docs, df_sec, df_chunks, pd.DataFrame(primeras), pd.DataFrame(leyendas), pd.DataFrame(tablas)


TIPOS_CLAVE = ["resumen", "introduccion", "objetivos", "area_estudio", "metodologia", "resultados", "conclusiones",
               "recomendaciones", "referencias"]


def reporte_extraccion(manifiesto, df_docs, df_sec, df_chunks, leyendas, tablas, n_docs, ocr_msg=""):
    ok, warn, err = "✅", "⚠️", "❌"
    chk = []
    estados = manifiesto["estado"].value_counts().to_dict()
    con_texto = estados.get("ok", 0) + estados.get("ok_ocr", 0)
    chk.append((ok if con_texto >= 0.9 * n_docs else warn,
                f"Documentos con texto del PDF: {con_texto}/{n_docs} (de ellos con OCR: {estados.get('ok_ocr', 0)})"))
    solo_meta = estados.get("solo_metadatos", 0)
    chk.append((ok if solo_meta == 0 else warn,
                f"Documentos solo con título + resumen (escaneados sin OCR): {solo_meta}"))
    sin_pdf = estados.get("sin_pdf", 0) + estados.get("error", 0)
    chk.append((ok if sin_pdf == 0 else err, f"Documentos sin PDF o con PDF dañado: {sin_pdf}"))
    sin_frag = n_docs - (df_chunks.paper_id.nunique() if len(df_chunks) else 0)
    chk.append((ok if sin_frag == 0 else err, f"Documentos sin ningún fragmento: {sin_frag}"))
    flags = manifiesto["flags"].fillna("")
    estructura = flags.str.contains("estructura_no_detectada").sum()
    chk.append((ok if estructura <= n_docs * 0.25 else warn,
                f"Documentos sin secciones detectadas (se fragmenta el cuerpo completo): {estructura}"))
    cobertura = {}
    for t in TIPOS_CLAVE:
        cobertura[t] = int(df_sec[df_sec.section_type == t].paper_id.nunique()) if len(df_sec) else 0
    if len(df_chunks):
        prior = (df_chunks.prioridad >= 2).sum()
        chk.append((ok if prior >= len(df_chunks) * 0.3 else warn,
                    f"Fragmentos con ≥2 pistas del dominio (prioritarios para la Persona 3): {prior}/{len(df_chunks)}"))
    idiomas = df_docs.idioma.value_counts().to_dict() if len(df_docs) else {}
    lineas = ["# Reporte de extracción de texto (Persona 2 · Glaciares)", "", f"Generado: {ahora()} · versión {VERSION}",
              "", f"OCR: {ocr_msg}", "", "| Estado | Verificación |", "|---|---|"] + [f"| {e} | {m} |" for e, m in chk]
    lineas += ["", "## Corpus procesado", "",
               f"- Páginas: {int(manifiesto.n_pages.fillna(0).sum())} · caracteres útiles: {int(manifiesto.n_chars_util.fillna(0).sum()):,}",
               f"- Idioma: {idiomas}",
               "- Por fuente: " + (", ".join(f"{k}: {v}" for k, v in df_docs.fuente.value_counts().items()) if len(df_docs) else ""),
               "- Documentos con cada sección: " + ", ".join(f"{k}: {v}" for k, v in cobertura.items())]
    if len(df_chunks):
        conteo_p = Counter(p for ps in df_chunks.pistas for p in ps)
        lineas += ["", "## Fragmentos (chunks)", "",
                   f"- Total: {len(df_chunks)} · promedio por documento: {len(df_chunks) / max(n_docs, 1):.1f}",
                   f"- Tamaño medio: {df_chunks.n_chars.mean():.0f} caracteres",
                   "- Por tipo de sección: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.section_type.value_counts().items()),
                   "- Por origen del texto: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.fuente_texto.value_counts().items()),
                   "- Fragmentos con cada pista: " + ", ".join(f"{k}: {v}" for k, v in conteo_p.most_common())]
    lineas += ["", "## Otros", "", f"- Leyendas (figuras, fotos, cuadros, mapas): {len(leyendas)}",
               f"- Tablas extraídas: {len(tablas)}", f"- Secciones totales: {len(df_sec)}"]
    problemas = manifiesto[flags != ""][["paper_id", "flags"]]
    if len(problemas):
        lineas += ["", "## Documentos con observaciones", "", "| paper_id | observaciones |", "|---|---|"]
        lineas += [f"| {r.paper_id} | {r.flags} |" for r in problemas.itertuples()]
    return chk, "\n".join(lineas), cobertura


def plantilla_anotacion(papers, n=12, semilla=7):
    """Muestra estratificada por fuente para anotar a mano (gold standard).
    Sirve para medir la extracción de la Persona 3 y adelanta la evaluación de la semana 12."""
    p = papers.copy()
    partes = [g.sample(min(len(g), max(1, n // p["fuente"].nunique())), random_state=semilla) for _, g in p.groupby("fuente")]
    m = pd.concat(partes)
    if len(m) < n:
        m = pd.concat([m, p[~p.paper_id.isin(m.paper_id)].sample(min(n - len(m), len(p) - len(m)), random_state=semilla)])
    cols = ["lagunas", "glaciares_nevados", "cordillera_cuenca_departamento", "nivel_de_riesgo", "eventos_y_fechas",
            "centros_poblados_expuestos", "obras_de_seguridad", "metodos_de_monitoreo", "instituciones",
            "mediciones_clave", "anotador", "notas"]
    salida = m[["paper_id", "title", "fuente"]].head(n).copy()
    for c in cols:
        salida[c] = ""
    return salida


README_TEXTO = """# Texto procesado del corpus Glaciares (Persona 2)

Entrada: `corpus_glaciares/papers.csv`, `pdfs/` y `pdfs_manifest.csv` (Persona 1).
Herramienta: PyMuPDF (tipografía y posición) + Tesseract para páginas escaneadas (opcional).

## Archivos

| Archivo | Contenido | Lo usa |
|---|---|---|
| `documentos_procesados.csv` | Una fila por documento: fuente, idioma, páginas, origen del texto (pdf, pdf+ocr, metadatos), secciones, fragmentos | Entrega del hito, control |
| `textos/<id>.txt` | Texto limpio completo de cada documento, con sus secciones | Lectura, revisión |
| `chunks.jsonl` (`.gz`) | Fragmentos de ~1500 caracteres con procedencia (documento, sección, páginas) y pistas del dominio | **Persona 3 (entidades y relaciones)** y RAG (semana 9) |
| `sections.csv` | Una fila por sección: tipo normalizado, título, páginas, texto | Persona 3, análisis |
| `tables.jsonl` | Tablas en Markdown (inventarios, áreas, volúmenes, cotas) | Persona 3 (valores numéricos) |
| `captions.csv` | Leyendas de figuras, fotos, cuadros y mapas | Persona 3 |
| `first_pages.jsonl` | Portada: instituciones candidatas, fechas | Persona 3 (publicadoPor, fechas) |
| `texto_manifest.csv` | Estado por PDF, hash, páginas escaneadas y con OCR, observaciones | Control de calidad |
| `reporte_extraccion.md` | Verificaciones automáticas | Entrega del hito |
| `anotacion_manual_plantilla.csv` | 12 documentos para anotar a mano (gold standard) | Evaluación de la extracción |

## Tipos de sección

`resumen, presentacion, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
desarrollo, discusion, conclusiones, recomendaciones, referencias, anexos, agradecimientos, indice,
front_matter, cuerpo, otro`

- `desarrollo`: capítulos propios del informe entre la introducción y las conclusiones
  (p. ej. "Descripción de los peligros", "Laguna Palcacocha").
- `cuerpo`: documento sin estructura detectable; se fragmenta completo.
- El índice, las referencias y los agradecimientos no se fragmentan.

## Campos de `chunks.jsonl`

`chunk_id` (`<documento>::s<sección>::c<n>`), `paper_id`, `fuente`, `title`, `year`, `section_idx`,
`section_type`, `heading`, `page_start`, `page_end`, `text`, `n_chars`, `pistas`, `prioridad`, `fuente_texto`.

- `pistas`: qué menciona el fragmento: `laguna`, `glaciar`, `peligro`, `poblado`, `obra`, `ubicacion`, `medicion`.
- `prioridad`: número de pistas (medición cuenta 0.5). Los fragmentos con prioridad ≥ 2 concentran los hechos
  que responden las preguntas de competencia; la Persona 3 puede empezar por ellos.
- `fuente_texto`: `pdf`, `ocr` (página escaneada leída con Tesseract) o `metadatos` (solo título y resumen).

## Uso de los textos

Los documentos son publicaciones de acceso abierto de instituciones públicas peruanas (INAIGEM, ANA, SENAMHI).
Se cita siempre la fuente con `paper_id` (handle del repositorio) y las páginas del fragmento.
"""


# -*- coding: utf-8 -*-
"""
Persona 3 — Entidades y relaciones (tripletas) desde el texto del corpus Glaciares
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG)

Entrada (Persona 2): texto_glaciares/chunks.jsonl (fragmentos con procedencia)
Varias técnicas combinadas, todas con evidencia (documento, páginas, oración o fila de tabla):
  1. Reglas y diccionarios del dominio (todo el corpus)
  2. Tablas de inventarios (valores limpios: área, volumen, altitud, ubicación)
  3. Diccionario de nombres construido con el propio corpus (encuentra "Palcacocha" sin la palabra "laguna")
  4. GLiNER multilingüe (nombres que las reglas no ven)
  5. spaCy: análisis gramatical sujeto → verbo → objeto (relaciones inesperadas)
  6. LLM (Gemini; Groq de respaldo) sobre una muestra, validado contra el texto
  + unificación de nombres, unidades normalizadas, reglas de sentido común y votación entre métodos
Salidas: entidades.csv, menciones.csv, tripletas_contenido.csv, tripletas_consolidadas.csv,
         tripletas_todas.csv (metadatos de la Persona 1 + contenido), reporte_pln.md
"""
import os, re, json, time, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone

import pandas as pd

VERSION = "2.0-glaciares"


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn")


def norm(t):
    return re.sub(r"\s+", " ", sin_tildes((t or "").lower())).strip()


def leer_jsonl(ruta):
    import gzip
    abrir = gzip.open if ruta.endswith(".gz") else open
    with abrir(ruta, "rt", encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]


# ======================================================================
# 1. Ontología de trabajo (tipos y predicados)
# ======================================================================
TIPOS = ["Laguna", "Glaciar", "Nevado", "Cordillera", "Cuenca", "Rio", "Quebrada", "Departamento", "Provincia",
         "Distrito", "CentroPoblado", "Evento", "Institucion", "MetodoMonitoreo", "ObraSeguridad",
         "Ecosistema", "Bofedal", "Fenomeno", "Especie", "ServicioEcosistemico"]

PREDICADOS = {
    "ubicadaEn": "lugar donde está una laguna, glaciar, nevado, cuenca, provincia o distrito",
    "alimentadaPor": "laguna alimentada por (o al pie de) un glaciar o nevado",
    "tieneNivelRiesgo": "nivel de riesgo o peligro de una laguna (Muy alto, Alto, Medio, Bajo)",
    "amenazaA": "laguna o glaciar que pone en peligro a un centro poblado aguas abajo",
    "ocurrioEn": "evento (aluvión, desborde, avalancha) ocurrido en una laguna o glaciar",
    "afectoA": "evento que afectó a un centro poblado",
    "ocurrioEnAnio": "año del evento",
    "tieneObraSeguridad": "obra de seguridad de una laguna (dique, túnel, sifón, canal de desagüe, sistema de alerta temprana)",
    "monitoreadoCon": "método con que se monitorea una laguna o glaciar",
    "intervieneEn": "institución que monitorea, evalúa o ejecuta obras en una laguna o glaciar",
    "tieneVolumen": "volumen de agua de una laguna (valor y unidad)",
    "tieneArea": "área o espejo de agua (valor y unidad)",
    "tieneAltitud": "altitud (m s.n.m.)",
    "tieneProfundidadMax": "profundidad máxima (m)",
    "tieneSuperficieGlaciar": "superficie glaciar de una cordillera (valor, unidad y año)",
    "estadoObra": "estado de una obra de seguridad de la laguna (p. ej. 'Dique: Requiere mantenimiento')",
    "formadaEnAnio": "año en que se formó o apareció una laguna",
    "causoVictimas": "número de víctimas (fallecidos) de un evento",
    "tienePoblacion": "número de habitantes de un centro poblado",
    "tieneEcosistema": "ecosistema presente en un lugar (bofedal, pajonal, bosque de Polylepis, humedal altoandino)",
    "presentaFenomeno": "fenómeno que ocurre en un lugar (retroceso glaciar, drenaje ácido de roca, contaminación por metales…)",
    "albergaEspecie": "especie de flora o fauna presente en un ecosistema o lugar",
    "proveeServicio": "servicio ecosistémico que brinda un ecosistema, glaciar o laguna (regulación hídrica, agua para consumo…)",
    "abasteceA": "laguna, río o glaciar que abastece de agua a un centro poblado o distrito",
    "drenaHacia": "laguna o río que desagua hacia un río, quebrada u otra laguna",
    "relacionadoCon": "relación abierta encontrada por el análisis gramatical (el verbo va en la columna 'verb')",
}

# ======================================================================
# 2. Diccionarios del dominio
# ======================================================================
CORDILLERAS = ["Blanca", "Negra", "Huallanca", "Huayhuash", "Raura", "Huagoruncho", "La Viuda", "Central",
               "Huaytapallana", "Chonta", "Ampato", "Urubamba", "Vilcabamba", "Huanzo", "Chila", "La Raya",
               "Vilcanota", "Carabaya", "Apolobamba", "Volcánica", "Barroso", "Real"]
DEPARTAMENTOS = ["Áncash", "Ancash", "Junín", "Cusco", "Cuzco", "Puno", "Huánuco", "Apurímac", "Huancavelica",
                 "Arequipa", "Ayacucho", "Pasco", "Cajamarca", "Moquegua", "Tacna", "Lima", "La Libertad", "Piura",
                 "Amazonas", "San Martín", "Ica", "Lambayeque", "Tumbes", "Loreto", "Ucayali", "Madre de Dios"]
DEP_CANON = {"ancash": "Áncash", "cuzco": "Cusco"}
DEPARTAMENTOS_SUELTOS = ["Áncash", "Ancash", "Junín", "Huánuco", "Apurímac", "Huancavelica", "Ayacucho", "Cajamarca",
                         "Moquegua", "Tacna", "Puno"]
POBLADOS_CONOCIDOS = ["Huaraz", "Carhuaz", "Yungay", "Caraz", "Recuay", "Huari", "Chavín", "Ranrahirca", "Mancos",
                      "Marcará", "Pomabamba", "Piscobamba", "Chacas", "Huallanca", "Ocongate", "Marcapata", "Macusani",
                      "Cabanaconde", "Chivay", "Urubamba", "Ollantaytambo", "Santa Teresa", "Quillabamba", "Huancayo",
                      "Paccha", "Llupa", "Pitec", "Cashapampa", "Tarica", "Aija", "Ticapampa", "Catac", "Cátac",
                      "Jangas", "Taricá", "Shilla", "Ataquero", "Musho", "Matacoto"]
INSTITUCIONES = {
    "INAIGEM": r"INAIGEM|Instituto Nacional de Investigaci[oó]n en Glaciares",
    "ANA": r"\bANA\b|Autoridad Nacional del Agua",
    "SENAMHI": r"SENAMHI|Servicio Nacional de Meteorolog[ií]a",
    "INDECI": r"INDECI|Instituto Nacional de Defensa Civil",
    "CENEPRED": r"CENEPRED",
    "INGEMMET": r"INGEMMET",
    "IGP": r"\bIGP\b|Instituto Geof[ií]sico del Per[uú]",
    "Electroperú": r"Electroper[uú]|ELECTROPER[UÚ]",
    "Hidrandina": r"Hidrandina|HIDRANDINA",
    "INRENA": r"INRENA",
    "Unidad de Glaciología": r"Unidad de Glaciolog[ií]a|UGRH",
    "Corporación Peruana del Santa": r"Corporaci[oó]n Peruana del Santa",
    "Comisión de Control de Lagunas": r"Comisi[oó]n de Control de Lagunas",
    "Gobierno Regional de Áncash": r"Gobierno Regional de [ÁA]ncash",
    "Municipalidad Provincial de Huaraz": r"Municipalidad Provincial de Huaraz",
}
METODOS = {
    "Batimetría": r"batim[eé]tr\w*|ecosonda",
    "Fotogrametría": r"fotogram[eé]tr\w*",
    "Dron (UAV)": r"\bdron(es)?\b|\bdrones\b|\bUAV\b|\bRPAS\b|veh[ií]culo a[eé]reo no tripulado",
    "Imágenes Landsat": r"\bLandsat\b",
    "Imágenes Sentinel": r"\bSentinel(-?\d\w?)?\b",
    "Teledetección": r"teledetecci[oó]n|sensores remotos|percepci[oó]n remota|im[aá]genes satelitales",
    "Georradar (GPR)": r"georradar|\bGPR\b|radar de penetraci[oó]n",
    "LiDAR": r"\bLiDAR\b",
    "GNSS / GPS diferencial": r"\bGNSS\b|GPS diferencial|\bDGPS\b",
    "Estación meteorológica": r"estaci[oó]n(es)? meteorol[oó]gica",
    "Limnígrafo / limnímetro": r"limn[ií]grafo|limn[ií]metro|sensor de nivel",
    "Balance de masa": r"balance de masa|mass balance",
    "Estacas de ablación": r"estacas? de ablaci[oó]n|ablation stakes",
    "InSAR / DInSAR": r"\bD?InSAR\b|interferometr\w*",
    "Modelamiento de flujo (HEC-RAS / FLO-2D / RAMMS)": r"HEC-?RAS|FLO-?2D|RAMMS|modelamiento hidr[aá]ulico|modelaci[oó]n num[eé]rica",
    "Cámaras de monitoreo": r"c[aá]maras? de (video)?vigilancia|c[aá]maras? de monitoreo|c[aá]mara time-?lapse",
}
OBRAS = {
    "Dique": r"\bdiques?\b|\bmuro de contenci[oó]n",
    "Túnel de desagüe": r"\bt[uú]nel(es)?\b",
    "Canal de desagüe": r"canal(es)? de (desag[uü]e|descarga|derivaci[oó]n|evacuaci[oó]n)",
    "Sifón": r"\bsif[oó]n(es)?\b",
    "Aliviadero": r"aliviadero",
    "Presa": r"\bpresa\b|\brepresa\b",
    "Sistema de alerta temprana": r"sistema de alerta temprana|\bSAT\b",
    "Obras de seguridad": r"obras? de seguridad",
}
ECOSISTEMAS = {
    "Bofedal": r"\bbofedal(es)?\b|\bwetlands? \(bofedal",
    "Pajonal altoandino": r"\bpajonal(es)?\b",
    "Bosque de Polylepis (queñual)": r"bosques? de (polylepis|que[ñn]u[ae]l(es)?|queu[ñn]a(s)?)|\bque[ñn]uales\b|\bqueu[ñn]ales\b|polylepis forests?",
    "Humedal altoandino": r"humedal(es)?( alto ?andinos?)?|\bpeatlands?\b|turberas?",
    "Puna": r"\bpuna\b",
}
FENOMENOS = {
    "Retroceso glaciar": r"retroceso (glaciar|del glaciar|de los glaciares)|p[eé]rdida de (superficie|cobertura|masa) glaciar|"
                         r"deglaciaci[oó]n|desglaciaci[oó]n|reducci[oó]n (de la )?(superficie|cobertura) glaciar|glacier retreat|glacier (mass )?loss",
    "Drenaje ácido de roca": r"drenaje [aá]cido( de roca)?|\bDAR\b|acid rock drainage|\bARD\b",
    "Contaminación por metales": r"metales pesados|contaminaci[oó]n (por|con) metales|trace[- ]metal|heavy metals",
    "Cambio climático": r"cambio clim[aá]tico|calentamiento global|climate change",
    "Escasez hídrica": r"escasez (h[ií]drica|de agua)|estr[eé]s h[ií]drico|d[eé]ficit h[ií]drico|water scarcity",
    "Aumento de temperatura": r"(aumento|incremento) de (la )?temperatura|warming",
    "Sequía": r"\bsequ[ií]as?\b|\bdrought",
    "Formación de nuevas lagunas": r"(formaci[oó]n|aparici[oó]n) de (nuevas )?lagunas|nuevas lagunas|new (glacial )?lakes",
}
ESPECIES = {
    "Polylepis (queñual)": r"\bPolylepis\b|\bque[ñn]ual\b|\bqueu[ñn]a\b",
    "Distichia muscoides": r"Distichia",
    "Plantago rigida": r"Plantago rigida",
    "Oxychloe andina": r"Oxychloe",
    "Puya raimondii": r"Puya raimondii|Puya Raimondi",
    "Ichu (Jarava ichu)": r"\bichu\b|Jarava ichu|Stipa ichu",
    "Vicuña": r"\bvicu[ñn]as?\b",
    "Alpaca": r"\balpacas?\b",
    "Llama": r"\bllamas\b",
    "Taruca": r"\btarucas?\b",
    "Cóndor andino": r"c[oó]ndor",
    "Vizcacha": r"\bvizcachas?\b",
    "Huallata (ganso andino)": r"huallata|Chloephaga",
    "Trucha": r"\btruchas?\b|Oncorhynchus",
    "Rana andina (Telmatobius)": r"Telmatobius",
}
SERVICIOS = {
    "Regulación hídrica": r"regulaci[oó]n (h[ií]drica|del agua|hidrol[oó]gica)|water regulation",
    "Provisión de agua": r"provisi[oó]n (de|del) agua|abastecimiento (de|del) agua|suministro de agua|water supply|fuente de agua",
    "Agua para consumo humano": r"consumo humano|agua potable",
    "Agua para riego": r"\briego\b|irrigaci[oó]n",
    "Generación hidroeléctrica": r"hidroel[eé]ctric\w*|hydropower",
    "Almacenamiento de carbono": r"(almacenamiento|captura|secuestro|reserva) de carbono|carbon (storage|stock|sequestration)",
    "Forraje y pastoreo": r"\bforraje\b|\bpastoreo\b|grazing",
    "Turismo": r"\bturis\w+|tourism",
    "Hábitat y biodiversidad": r"h[aá]bitat|biodiversidad|biodiversity",
}
ESTADOS_OBRA = [
    ("Colapsada", r"colaps\w+|destruid[ao]s?"),
    ("Deteriorada", r"deteriorad[ao]s?|en mal estado|mal estado|da[ñn]ad[ao]s?|fisurad[ao]s?|agrietad[ao]s?|erosionad[ao]s?|obstruid[ao]s?|colmatad[ao]s?"),
    ("Requiere mantenimiento", r"requiere[n]? (de )?mantenimiento|necesita[n]? mantenimiento|falta de mantenimiento|sin mantenimiento"),
    ("Inoperativa", r"inoperativ[ao]s?|no funciona|fuera de servicio|abandonad[ao]s?"),
    ("En construcción", r"en construcci[oó]n|en ejecuci[oó]n|por construir|proyectad[ao]s?"),
    ("Rehabilitada", r"rehabilitad[ao]s?|reconstruid[ao]s?|reparad[ao]s?"),
    ("Operativa", r"operativ[ao]s?|en buen estado|buen estado|en funcionamiento|funciona(n)? (adecuadamente|correctamente)|en buenas condiciones"),
]
PAT_FORMADA = re.compile(r"(?:se form[oó]|formad[ao]|apareci[oó]|surgi[oó]|se origin[oó]|originad[ao]|was formed|formed in|appeared in)"
                         r"[^.;]{0,50}?\b(19\d{2}|20[0-4]\d)\b", re.I)
PAT_VICTIMAS = re.compile(r"(?:muerte de|murieron|fallecieron|perdieron la vida|v[ií]ctimas[^.;]{0,25}?|deaths? of|killed|killing)\s*"
                          r"(?:aproximadamente |unas? |cerca de |m[aá]s de |alrededor de |about |more than |over |some )?"
                          r"(?P<num>\d{1,3}(?:[ .,]\d{3})+|\d{2,6})\b", re.I)
PAT_POBLACION = re.compile(r"(?P<num>\d{1,3}(?:[ .,]\d{3})+|\d{3,7})\s*(?:habitantes|pobladores|inhabitants|residents)", re.I)

EVENTOS = r"(aluvi[oó]n|desborde|avalancha|\balud\b|GLOF|huaico|huayco|deslizamiento|desembalse|rotura de (?:la )?morrena)"
# desborde, GLOF, rotura de morrena y desembalse súbito son el mismo tipo de evento: aluvión (inundación por desborde)
EVENTO_CANON = {"aluvion": "Aluvión", "desborde": "Aluvión", "avalancha": "Avalancha", "alud": "Avalancha",
                "glof": "Aluvión", "huaico": "Huaico", "huayco": "Huaico", "deslizamiento": "Deslizamiento",
                "desembalse": "Aluvión", "rotura de morrena": "Aluvión", "rotura de la morrena": "Aluvión"}
PAT_CITA = re.compile(r"(?:\bet al\.?,?|[A-ZÁÉÍÓÚ][a-záéíóúñ]+,|[A-ZÁÉÍÓÚ][a-záéíóúñ]+ (?:y|and|&) [A-ZÁÉÍÓÚ][a-záéíóúñ]+,?)\s*\(?(?:19|20)\d{2}[a-z]?\)?"
                      r"|\((?:[^()]*?,\s*)?(?:19|20)\d{2}[a-z]?(?:;[^()]*)?\)")

# palabras con mayúscula que NO son nombres propios
NO_NOMBRES = set(norm(x) for x in """la el los las lo este esta estos estas dicha dicho dichas dichos su sus en se que con por
para del de una un al y o segun durante desde hasta ademas asimismo sin fuente figura foto fotografia cuadro tabla mapa
glaciar glaciares laguna lagunas nevado nevados cordillera cuenca rio quebrada n no nro y/o total ver anexo sector zona
area parte grupo sistema proyecto estudio informe evaluacion inventario monitoreo peligro riesgo alto medio bajo muy
norte sur este oeste superior inferior principal nacional regional general tipo nivel volumen fecha ano anos
ubicacion caracteristicas resultados conclusiones recomendaciones objetivo objetivos metodologia introduccion
of the and in peru mundo andes america sudamerica latinoamerica origen alto-andinas altoandinas alto altas montana
tropicales tropical inventario nacional mayor menor gran grandes pequenas cabecera tipo formacion lake lakes glacier
glaciers city town peruvian andean the these those many most all some other debris rock small large
mountain outlet cirque valley tropical proglacial moraine-dammed glacial a an this that
vista imagen detalle panoramica elaboracion leyenda croquis plano esquema perfil corte seccion""".split())

LETRA = "A-ZÁÉÍÓÚÑÜ"
PAL = rf"[{LETRA}][A-Za-zÁÉÍÓÚÑÜáéíóúñü'\-]+"
SUFIJO = r"(?:Alta|Baja|Grande|Chico|Chica|Nueva|Vieja|Norte|Sur|Este|Oeste|Superior|Inferior|I{1,3}\b|IV\b|\d{1,3}(?![\d.,]))"
NOMBRE = rf"(?:{PAL}|\d{{2,3}}(?!\d))(?:\s+{SUFIJO})?"
LISTA = rf"{NOMBRE}(?:\s*(?:,|\by\b|\be\b)\s*(?:(?:la|el)\s+)?{NOMBRE})*"


def _limpio(nombre):
    n = re.sub(r"\s+", " ", nombre).strip(" ,.;:()-")
    if not n or len(n) < 2 or len(n) > 60:
        return ""
    if norm(n) in NO_NOMBRES or norm(n.split()[0]) in NO_NOMBRES:
        return ""
    if n.isupper() and len(n) > 3:
        n = " ".join(w.capitalize() if not re.fullmatch(r"I{1,3}|IV", w) else w for w in n.split())
    return n


def _lista(s):
    partes = re.split(r"\s*(?:,|\by\b|\be\b)\s*", s)
    return [p for p in (_limpio(re.sub(r"^(la|el)\s+", "", x)) for x in partes) if p]


def clave(tipo, nombre):
    return f"{tipo}:{re.sub(r'[^a-z0-9]+', '_', norm(nombre)).strip('_')}"


def canon_nombre(tipo, nombre):
    n = re.sub(r"^(la|el|los|las)\s+", "", nombre.strip(), flags=re.I)
    prefijos = {
        "Laguna": r"^(lagunas?|lakes?)\s+(de\s+|del\s+|of\s+)?", "Bofedal": r"^bofedal(es)?\s+(de\s+|del\s+)?", "Glaciar": r"^(glaciar(es)?|glaciers?)\s+(de\s+|del\s+|of\s+)?",
        "Nevado": r"^(nevados?|mount|mt\.?|cerro)\s+(de\s+|del\s+)?", "Cordillera": r"^(cordillera|cordilleras)\s+",
        "Cuenca": r"^(sub|micro)?cuenca\s+(hidrogr[aá]fica\s+)?(de(l)?\s+)?((r[ií]o|quebrada)\s+)?",
        "Rio": r"^r[ií]o\s+", "Quebrada": r"^quebrada\s+", "Provincia": r"^provincia\s+(de\s+)?",
        "Distrito": r"^distrito\s+(de\s+)?", "Departamento": r"^(departamento|regi[oó]n)\s+(de\s+)?",
        "CentroPoblado": r"^(ciudad|centro poblado|localidad|caser[ií]o|poblado|pueblo|comunidad campesina|city|town|village)\s+(de\s+|of\s+)?",
    }
    sufijos = {"Laguna": r"\s+(lake|laguna)$", "Glaciar": r"\s+(glacier|glaciar)$", "Nevado": r"\s+(peak|mountain)$",
               "Cordillera": r"\s+(range|mountains)$"}
    if tipo in prefijos:
        n = re.sub(prefijos[tipo], "", n, flags=re.I)
    if tipo in sufijos:
        n = re.sub(sufijos[tipo], "", n, flags=re.I)
    if tipo == "Institucion":
        for canon, p in INSTITUCIONES.items():
            if re.fullmatch(r"(?:" + p + r")", n.strip()):
                return canon
        if not (re.fullmatch(r"[A-ZÁÉÍÓÚÑ]{2,10}", n.strip()) or re.search(
                r"instituto|universi|ministerio|autoridad|servicio|direcci[oó]n|municipalidad|gobierno|agencia|"
                r"comisi[oó]n|corporaci[oó]n|empresa|programa|proyecto|centro|unidad|organizaci|fundaci|asociaci|"
                r"institute|university|agency|department|survey|council|foundation", n, re.I)):
            return ""
    n = _limpio(n) if tipo not in ("Institucion", "MetodoMonitoreo", "ObraSeguridad", "Evento", "Ecosistema",
                                   "Fenomeno", "Especie", "ServicioEcosistemico") else n.strip()
    if tipo == "Bofedal" and n and (len(n.split()) > 3 or re.search(
            r"bosque|p[aá]ramo|batolito|tundra|pajonal|zona|formaci|montano|tropical|h[uú]med|subalpin|piso|vegetaci|"
            r"pradera|matorral|estepa|desierto|puna|herbazal|c[eé]sped", n, re.I)):
        return ""
    if tipo in ("Laguna", "Glaciar", "Nevado") and norm(n) in {"blanca", "negra", "central", "real", "volcanica"}:
        return ""   # "glaciares de la Blanca" → es la cordillera, no un glaciar
    if tipo == "Departamento" and n:
        n = DEP_CANON.get(norm(n), n)
    return n


# ======================================================================
# 3. Oraciones y menciones por reglas
# ======================================================================
PAT_ORACION = re.compile(r"[^.!?;\n]+(?:[.!?;]+|$)|\n+")


def oraciones(texto):
    """Lista de (inicio, fin, oración). Corta en . ! ? ; y saltos de párrafo (no en '17,4' ni en abreviaturas)."""
    texto = texto or ""
    out, ini = [], 0
    for m in re.finditer(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÑ¿¡(\d])|\n{2,}", texto):
        fin = m.start()
        frag = texto[ini:fin]
        if re.search(r"(aprox|Fig|N°|Nº|Sr|Dr|Ing|et al|Ap|Cap)\.$", frag.strip()):
            continue
        if frag.strip():
            out.append((ini, fin, frag))
        ini = m.end()
    if texto[ini:].strip():
        out.append((ini, len(texto), texto[ini:]))
    return out


def _re(p, flags=0):
    return re.compile(p, flags)


REGLAS_ENTIDADES = [
    ("Laguna", _re(rf"\b(?:[Ll]agunas?|LAGUNAS?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Glaciar", _re(rf"\b(?:[Gg]laciar(?:es)?|GLACIAR(?:ES)?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Nevado", _re(rf"\b(?:[Nn]evados?|NEVADOS?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    # artículos en inglés
    ("Laguna", _re(rf"\bLakes?\s+(?P<n>{LISTA})")),
    ("Laguna", _re(rf"\b(?P<n>{PAL})\s+[Ll]akes?\b")),
    ("Glaciar", _re(rf"\b(?P<n>{PAL})\s+[Gg]laciers?\b")),
    ("Nevado", _re(rf"\b(?:Mount|Mt\.|Nevado)\s+(?P<n>{PAL})")),
    ("Bofedal", _re(rf"\b(?:[Bb]ofedal(?:es)?|BOFEDAL(?:ES)?)\s+(?:(?:de|del)\s+)?(?P<n>{LISTA})")),
    ("Cordillera", _re(r"\b[Cc]ordillera(?:s)?\s+(?P<n>" + "|".join(sorted(CORDILLERAS, key=len, reverse=True)) + r")\b")),
    ("Cuenca", _re(rf"\b(?:[Ss]ub|[Mm]icro)?[Cc]uencas?\s+(?:hidrogr[aá]fica\s+)?(?:(?:de|del)\s+)?(?:(?:[Rr][ií]o|[Qq]uebrada)\s+)?(?P<n>{PAL}(?:\s+{PAL})?)")),
    ("Rio", _re(rf"\b[Rr][ií]o\s+(?P<n>{PAL})")),
    ("Quebrada", _re(rf"\b[Qq]uebrada\s+(?:(?:de|del)\s+)?(?P<n>{PAL})")),
    ("Provincia", _re(rf"\b[Pp]rovincias?\s+de\s+(?P<n>{PAL}(?:\s+{PAL})?)")),
    ("Distrito", _re(rf"\b[Dd]istritos?\s+de\s+(?P<n>{PAL}(?:\s+{PAL})?)")),
    ("Departamento", _re(r"\b(?:[Dd]epartamento|[Rr]egi[oó]n)\s+(?:de\s+)?(?P<n>" + "|".join(sorted(DEPARTAMENTOS, key=len, reverse=True)) + r")\b")),
    ("Departamento", _re(r"\b(?P<n>" + "|".join(DEPARTAMENTOS_SUELTOS) + r")\b")),
    ("CentroPoblado", _re(rf"\b(?:[Cc]iudad|[Cc]entro [Pp]oblado|[Ll]ocalidad|[Cc]aser[ií]o|[Pp]oblado|[Pp]ueblo|[Cc]omunidad [Cc]ampesina)\s+(?:de\s+)?(?P<n>{PAL}(?:\s+{PAL})?)")),
    ("CentroPoblado", _re(r"\b(?P<n>" + "|".join(sorted(POBLADOS_CONOCIDOS, key=len, reverse=True)) + r")\b")),
]


def menciones_reglas(texto):
    """Menciones (tipo, nombre canónico, inicio, fin, metodo) encontradas por reglas."""
    out = []
    for tipo, pat in REGLAS_ENTIDADES:
        for m in pat.finditer(texto):
            grupo = m.group("n")
            nombres = _lista(grupo) if tipo in ("Laguna", "Glaciar", "Nevado", "Bofedal") else [grupo]
            pos = m.start("n")
            for n in nombres:
                c = canon_nombre(tipo, n)
                if c:
                    i = texto.find(n, pos)
                    i = i if i >= 0 else m.start("n")
                    out.append({"tipo": tipo, "nombre": c, "ini": i, "fin": i + len(n), "metodo": "reglas"})
    for nombre, p in INSTITUCIONES.items():
        for m in re.finditer(p, texto):
            out.append({"tipo": "Institucion", "nombre": nombre, "ini": m.start(), "fin": m.end(), "metodo": "reglas"})
    for nombre, p in METODOS.items():
        for m in re.finditer(p, texto, re.I):
            out.append({"tipo": "MetodoMonitoreo", "nombre": nombre, "ini": m.start(), "fin": m.end(), "metodo": "reglas"})
    for nombre, p in OBRAS.items():
        for m in re.finditer(p, texto, re.I if nombre != "Sistema de alerta temprana" else 0):
            out.append({"tipo": "ObraSeguridad", "nombre": nombre, "ini": m.start(), "fin": m.end(), "metodo": "reglas"})
    for tipo, dic, flags in (("Ecosistema", ECOSISTEMAS, re.I), ("Fenomeno", FENOMENOS, re.I),
                             ("Especie", ESPECIES, 0), ("ServicioEcosistemico", SERVICIOS, re.I)):
        for nombre, p in dic.items():
            for m in re.finditer(p, texto, flags):
                out.append({"tipo": tipo, "nombre": nombre, "ini": m.start(), "fin": m.end(), "metodo": "reglas"})
    # quitar duplicados y menciones contenidas en otra del mismo tipo
    out.sort(key=lambda x: (x["ini"], -(x["fin"] - x["ini"])))
    final = []
    for x in out:
        if any(y["tipo"] == x["tipo"] and y["ini"] <= x["ini"] and x["fin"] <= y["fin"] and y["nombre"] == x["nombre"]
               for y in final):
            continue
        final.append(x)
    return final


# ======================================================================
# 4. GLiNER (opcional)
# ======================================================================
ETIQUETAS_GLINER = {
    "laguna": "Laguna", "glaciar": "Glaciar", "nevado o montaña": "Nevado", "cordillera": "Cordillera",
    "río": "Rio", "cuenca hidrográfica": "Cuenca", "ciudad o centro poblado": "CentroPoblado",
    "institución u organización": "Institucion",
}


ETIQUETAS_GLINER_V2 = {"bofedal o humedal altoandino": "Bofedal", "especie de planta o animal": "Especie"}
PAT_TEMAS_V2 = re.compile(r"bofedal|humedal|turbera|especie|flora|fauna|vegetaci[oó]n|wetland|species", re.I)


def cargar_gliner(modelo="urchade/gliner_multi-v2.1"):
    import warnings
    warnings.filterwarnings("ignore", message=".*truncated to 384.*")
    warnings.filterwarnings("ignore", category=FutureWarning)
    from gliner import GLiNER
    import torch
    m = GLiNER.from_pretrained(modelo)
    disp = "cuda" if torch.cuda.is_available() else "cpu"
    m = m.to(disp)
    return m, disp


def ventanas(texto, maximo=900):
    """Divide el texto en ventanas por oraciones (GLiNER lee ~384 palabras)."""
    out, ini, actual_ini, actual_fin = [], 0, None, None
    for i, f, _ in oraciones(texto):
        if actual_ini is None:
            actual_ini, actual_fin = i, f
        elif f - actual_ini > maximo:
            out.append((actual_ini, actual_fin))
            actual_ini, actual_fin = i, f
        else:
            actual_fin = f
    if actual_ini is not None:
        out.append((actual_ini, actual_fin))
    return out


def menciones_gliner(modelo, chunks, umbral=0.5, lote=8, cache_ruta=None, log=print, etiquetas_map=None):
    """Menciones de GLiNER por chunk_id. Reanudable con un caché jsonl."""
    hechos = {}
    if cache_ruta and os.path.exists(cache_ruta):
        for r in leer_jsonl(cache_ruta):   # se vuelven a normalizar (por si cambiaron las reglas de nombres)
            ms = []
            for m in r["menciones"]:
                n = canon_nombre(m["tipo"], m["nombre"])
                if n:
                    ms.append({**m, "nombre": n})
            hechos[r["chunk_id"]] = ms
    pendientes = [c for c in chunks if c["chunk_id"] not in hechos]
    log(f"GLiNER: {len(hechos)} fragmentos en caché, {len(pendientes)} por procesar")
    etiquetas_map = etiquetas_map or ETIQUETAS_GLINER
    etiquetas = list(etiquetas_map)
    fh = open(cache_ruta, "a", encoding="utf-8") if cache_ruta else None
    t0 = time.time()
    try:
        for k in range(0, len(pendientes), 64):
            bloque = pendientes[k:k + 64]
            textos, idx = [], []
            for c in bloque:
                for (i, f) in ventanas(c["text"]):
                    textos.append(c["text"][i:f]); idx.append((c["chunk_id"], i))
            preds = []
            for j in range(0, len(textos), lote):
                preds += modelo.batch_predict_entities(textos[j:j + lote], etiquetas, threshold=umbral)
            por_chunk = defaultdict(list)
            for (cid, desplaz), ents in zip(idx, preds):
                for e in ents:
                    tipo = etiquetas_map.get(e["label"])
                    nombre = canon_nombre(tipo, e["text"]) if tipo else ""
                    if tipo == "Especie":   # solo nombres científicos (Género especie) o especies conocidas
                        conocida = next((k for k, p in ESPECIES.items() if re.search(p, e["text"], re.I)), None)
                        nombre = conocida or (nombre if re.fullmatch(r"[A-Z][a-z]{3,} [a-z]{3,}", nombre) else "")
                    if not nombre or not re.search(rf"[{LETRA}0-9]", nombre):
                        continue
                    por_chunk[cid].append({"tipo": tipo, "nombre": nombre, "ini": desplaz + e["start"],
                                           "fin": desplaz + e["end"], "metodo": "gliner",
                                           "score": round(float(e.get("score", 0)), 3)})
            for c in bloque:
                hechos[c["chunk_id"]] = por_chunk.get(c["chunk_id"], [])
                if fh:
                    fh.write(json.dumps({"chunk_id": c["chunk_id"], "menciones": hechos[c["chunk_id"]]}, ensure_ascii=False) + "\n")
            if fh:
                fh.flush()
            hechos_n = k + len(bloque)
            vel = hechos_n / max(time.time() - t0, 1e-6)
            log(f"   GLiNER {hechos_n}/{len(pendientes)} · {vel:.1f} frag/s · faltan ~{(len(pendientes) - hechos_n) / max(vel, 1e-6) / 60:.0f} min")
    finally:
        if fh:
            fh.close()
    return hechos


# ======================================================================
# 5. Relaciones por reglas (en cada oración, con la mención más cercana)
# ======================================================================
T_UBIC = r"ubicad|situad|se encuentra|se localiza|localizad|pertenec|forma parte|en la cordillera|en la (sub|micro)?cuenca|en el distrito|en la provincia"
T_ALIM = r"al pie|alimenta|deshielo|frente glaciar|lengua glaciar|lengua del|origen glaciar|retroceso del glaciar|desprend|bajo el glaciar|debajo del|aporte glaciar|derivad"
T_AMEN = r"aguas abajo|expuest|amenaz|en riesgo|en peligro|afectar[ií]a|podr[ií]a afectar|vulnerab|poner en riesgo|zona de impacto|recorrido del flujo"
T_AFEC = r"afect|destru|arras|sepult|v[ií]ctimas|muert|fallec|da[ñn]|inund|devast"
T_INTERV = r"monitore|inspeccion|evalu|vigil|ejecut|constru|instal|realiz|estudi|supervis|implement"
PAT_RIESGO = re.compile(r"(?:riesgo|peligro(?:sidad)?|amenaza)\s+(?:(?:es|ser[ií]a|era|fue|se considera|considerado como|calificado como|de nivel)\s+)?"
                        r"(?P<v>muy\s+alto|alto|medio|moderado|bajo|muy\s+bajo|cr[ií]tico)\b", re.I)
NUM = r"(?P<num>\d{1,3}(?:[ .]\d{3})*(?:,\d+)?|\d+(?:[.,]\d+)?)"
PAT_MEDIDAS = [
    ("tieneVolumen", re.compile(r"volumen[^.;]{0,70}?" + NUM + r"\s*(?P<u>hm3|hm³|MMC|MMm3|millones de m3|millones de metros c[uú]bicos|m3|m³)", re.I)),
    ("tieneArea", re.compile(r"(?:[áa]rea|espejo de agua|superficie)(?! glaciar)[^.;]{0,60}?" + NUM + r"\s*(?P<u>km2|km²|m2|m²|ha)\b", re.I)),
    ("tieneAltitud", re.compile(r"(?:altitud|cota|a una altura|ubicad[oa] a|situad[oa] a|,\s*a)[^.;]{0,40}?" + NUM + r"\s*(?P<u>m\s?\.?\s?s\.?\s?n\.?\s?m\.?|msnm)", re.I)),
    ("tieneProfundidadMax", re.compile(r"profundidad m[aá]xima[^.;]{0,40}?" + NUM + r"\s*(?P<u>m)\b", re.I)),
]
PAT_SUP_GLACIAR = re.compile(r"(?:superficie|cobertura|[áa]rea) glaciar[^.;]{0,80}?" + NUM + r"\s*(?P<u>km2|km²|ha)\b", re.I)
PAT_ANIO = re.compile(r"\b(1[89]\d{2}|20[0-4]\d)\b")


def numero(s):
    s = s.strip().replace(" ", "")
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def _cercano(x, candidatos):
    return min(candidatos, key=lambda y: abs(y["ini"] - x["ini"])) if candidatos else None


def _metodo_par(s, ob):
    ms = {s.get("metodo", "reglas")} | ({ob.get("metodo", "reglas")} if isinstance(ob, dict) else set())
    if ms <= {"reglas", "tabla"}:
        return "reglas"
    if "diccionario" in ms and "gliner" not in ms:
        return "diccionario"
    return "gliner+reglas"


def relaciones_oracion(o, ments, base, foco=()):
    """Tripletas de una oración. ments: menciones dentro de la oración (posiciones relativas al chunk).
    foco: lagunas de la oración anterior o del título (para "la laguna", "el volumen"... sin nombre)."""
    t = []
    ini, fin, texto = o
    tl = texto.lower()
    por = defaultdict(list)
    for m in ments:
        por[m["tipo"]].append(m)

    sin_citas = PAT_CITA.sub(lambda x: " " * len(x.group(0)), texto)   # (Worni et al., 2014) no es el año del evento
    anios_or = PAT_ANIO.findall(sin_citas)
    anio_or = anios_or[-1] if anios_or else ""

    def add(s, p, ob, conf, tipo_ob=None, valor=None, unidad="", anio=""):
        t.append({**base, "subject": s["nombre"], "subject_type": s["tipo"], "predicate": p, "year": anio,
                  "object": ob["nombre"] if isinstance(ob, dict) else ob,
                  "object_type": ob["tipo"] if isinstance(ob, dict) else (tipo_ob or "Literal"),
                  "value": valor, "unit": unidad, "evidence": texto.strip()[:600], "confidence": conf,
                  "method": _metodo_par(s, ob)})

    lugares = ["Cordillera", "Cuenca", "Quebrada", "Departamento", "Provincia", "Distrito"]
    trig_ubic = bool(re.search(T_UBIC, tl))
    sin_nombre = (not por["Laguna"] and foco and trig_ubic and re.search(r"\bla laguna\b|\bdicha laguna\b|\besta laguna\b", tl))
    for tipo_s in ("Laguna", "Glaciar", "Nevado"):
        for s in (por[tipo_s] if not (tipo_s == "Laguna" and sin_nombre) else list(foco)):
            for tl_ in lugares:
                ob = _cercano(s, por[tl_])
                if ob:
                    add(s, "ubicadaEn", ob, 0.8 if trig_ubic else 0.6)
    for s in por["Provincia"] + por["Distrito"] + por["Cuenca"]:
        for tl_ in (["Departamento"] if s["tipo"] != "Distrito" else ["Provincia", "Departamento"]):
            ob = _cercano(s, por[tl_])
            if ob and not por["Laguna"]:
                add(s, "ubicadaEn", ob, 0.7)
    if re.search(T_ALIM, tl):
        for s in por["Laguna"]:
            ob = _cercano(s, por["Glaciar"] + por["Nevado"])
            if ob:
                add(s, "alimentadaPor", ob, 0.7)
    for m in PAT_RIESGO.finditer(texto):
        v = re.sub(r"\s+", " ", m.group("v").lower())
        v = {"moderado": "Medio", "critico": "Muy alto", "crítico": "Muy alto"}.get(v, v.capitalize())
        pseudo = {"ini": ini + m.start()}
        lags = {x["nombre"]: x for x in por["Laguna"]}
        if len(lags) > 1 and re.search(r"\blagunas\b", tl) and len(lags) <= 6:
            for s in lags.values():
                add(s, "tieneNivelRiesgo", v, 0.7)
            continue
        s = _cercano(pseudo, por["Laguna"] or por["Glaciar"] + por["Nevado"])
        if s:
            add(s, "tieneNivelRiesgo", v, 0.75)
        elif foco and re.search(r"\blaguna\b|\bla misma\b|\bdicha\b", tl):
            for s in foco:
                add(s, "tieneNivelRiesgo", v, 0.6)
    if re.search(T_AMEN, tl):
        sujetos = por["Laguna"] + por["Glaciar"]
        for s in (sujetos or list(foco)):
            for ob in por["CentroPoblado"]:
                add(s, "amenazaA", ob, 0.7 if sujetos else 0.6)
    # Eventos con año
    for m in re.finditer(EVENTOS, texto, re.I):
        anios = [a for a in PAT_ANIO.finditer(sin_citas) if abs(a.start() - m.start()) < 90]
        if not anios:
            continue
        anio = min(anios, key=lambda a: abs(a.start() - m.start())).group(1)
        tipo_ev = EVENTO_CANON.get(norm(m.group(1)), m.group(1).capitalize())
        lag = _cercano({"ini": ini + m.start()}, por["Laguna"] + por["Glaciar"] + por["Nevado"])
        nombre = f"{tipo_ev} {lag['nombre']} {anio}" if lag else f"{tipo_ev} {anio}"
        ev = {"tipo": "Evento", "nombre": nombre, "ini": ini + m.start(), "metodo": "reglas"}
        add(ev, "ocurrioEnAnio", anio, 0.8, "Literal", float(anio), anio=anio)
        if lag:
            add(ev, "ocurrioEn", lag, 0.8)
        for v in PAT_VICTIMAS.finditer(texto):
            n = numero(v.group("num"))
            if n and not (1800 <= n <= 2049 and len(v.group("num")) == 4):
                add(ev, "causoVictimas", v.group("num").strip(), 0.7, "Literal", n, "personas", anio)
        for p in por["CentroPoblado"]:
            add(ev, "afectoA", p, 0.75 if re.search(T_AFEC, tl) else 0.55)
    for s in (por["Laguna"] or list(foco)):
        for ob in {o_["nombre"]: o_ for o_ in por["ObraSeguridad"]}.values():
            add(s, "tieneObraSeguridad", ob["nombre"], 0.7 if por["Laguna"] else 0.6, "ObraSeguridad")
    for s in por["Laguna"] + por["Glaciar"] + por["Nevado"]:
        for ob in {o_["nombre"]: o_ for o_ in por["MetodoMonitoreo"]}.values():
            add(s, "monitoreadoCon", ob["nombre"], 0.6, "MetodoMonitoreo")
    if re.search(T_INTERV, tl):
        for s in {i_["nombre"]: i_ for i_ in por["Institucion"]}.values():
            for ob in por["Laguna"] + por["Glaciar"]:
                add(s, "intervieneEn", ob, 0.6)
    # Mediciones
    for pred, pat in PAT_MEDIDAS:
        for m in pat.finditer(texto):
            s = _cercano({"ini": ini + m.start()}, por["Laguna"] or (por["Glaciar"] if pred != "tieneVolumen" else []))
            if not s and foco and not (por["Glaciar"] or por["Nevado"] or por["Cordillera"]):
                s = foco[0]
            if s:
                u = re.sub(r"\s+", "", m.group("u")).replace("³", "3").replace("²", "2")
                u = "m s.n.m." if "s" in u.lower() and "n" in u.lower() else u
                obj = f"{m.group('num').strip()} {u}" + (f" ({anio_or})" if anio_or and pred != "tieneAltitud" else "")
                add(s, pred, obj, 0.7, "Literal", numero(m.group("num")), u, anio_or if pred != "tieneAltitud" else "")
    for m in PAT_SUP_GLACIAR.finditer(texto):
        s = _cercano({"ini": ini + m.start()}, por["Cordillera"])
        if s:
            anios = PAT_ANIO.findall(texto)
            u = m.group("u").replace("²", "2")
            add(s, "tieneSuperficieGlaciar", f"{m.group('num').strip()} {u}" + (f" ({anios[-1]})" if anios else ""),
                0.65, "Literal", numero(m.group("num")), u, anios[-1] if anios else "")

    # ---- v1.1: estado de obras, formación, población, ecosistemas, fenómenos, especies, servicios ----
    obras = por["ObraSeguridad"]
    if obras:
        for estado, pat in ESTADOS_OBRA:
            for m in re.finditer(pat, texto, re.I):
                ob = _cercano({"ini": ini + m.start()}, obras)
                for s in (por["Laguna"] or list(foco)):
                    add(s, "estadoObra", f"{ob['nombre']}: {estado}", 0.65 if por["Laguna"] else 0.55, "Literal", anio=anio_or)
                break
    for m in PAT_FORMADA.finditer(sin_citas):
        s = _cercano({"ini": ini + m.start()}, por["Laguna"]) or (foco[0] if foco and re.search(r"laguna|lake", tl) else None)
        if s:
            add(s, "formadaEnAnio", m.group(1), 0.7, "Literal", float(m.group(1)), anio=m.group(1))
    if not re.search(r"muert|murieron|fallec|v[ií]ctimas|killed|deaths", tl):
        for m in PAT_POBLACION.finditer(texto):
            p = _cercano({"ini": ini + m.start()}, [x for x in por["CentroPoblado"] if x["ini"] <= ini + m.start()] or por["CentroPoblado"])
            if p:
                add(p, "tienePoblacion", m.group("num").strip(), 0.6, "Literal", numero(m.group("num")), "habitantes", anio_or)
    lugares_amplios = ["Laguna", "Cuenca", "Quebrada", "Distrito", "Provincia", "Departamento", "Cordillera", "Bofedal"]
    for e in {x["nombre"]: x for x in por["Ecosistema"]}.values():
        for tl_ in lugares_amplios:
            ob = _cercano(e, por[tl_])
            if ob:
                add(ob, "tieneEcosistema", e, 0.6)
    for b in por["Bofedal"]:
        for tl_ in ["Cuenca", "Quebrada", "Distrito", "Provincia", "Departamento", "Cordillera"]:
            ob = _cercano(b, por[tl_])
            if ob:
                add(b, "ubicadaEn", ob, 0.7)
    for f in {x["nombre"]: x for x in por["Fenomeno"]}.values():
        for tl_ in ["Glaciar", "Nevado", "Laguna", "Cuenca", "Quebrada", "Cordillera", "Distrito", "Provincia", "Departamento"]:
            ob = _cercano(f, por[tl_])
            if ob:
                add(ob, "presentaFenomeno", f, 0.6)
    sujetos_eco = por["Ecosistema"] + por["Bofedal"]
    for esp in {x["nombre"]: x for x in por["Especie"]}.values():
        for s in ({x["nombre"]: x for x in sujetos_eco}.values() if sujetos_eco else
                  [ _cercano(esp, por[tl_]) for tl_ in ("Laguna", "Cuenca", "Quebrada", "Distrito") if por[tl_]]):
            add(s, "albergaEspecie", esp, 0.6)
    if re.search(r"servicio|provee|brinda|aporta|regula|almacen|abastec|importan|beneficio|provid|supply|service", tl):
        for sv in {x["nombre"]: x for x in por["ServicioEcosistemico"]}.values():
            for s in {x["nombre"]: x for x in sujetos_eco + por["Glaciar"] + por["Laguna"]}.values():
                add(s, "proveeServicio", sv, 0.55)
    return t


def extraer_chunk(c, menc_extra=None):
    """Menciones y tripletas de un fragmento. menc_extra: menciones de GLiNER."""
    texto = c["text"]
    menc = menciones_reglas(texto)
    if menc_extra:
        ya = {(m["tipo"], norm(m["nombre"]), m["ini"]) for m in menc}
        cubiertos = [(m["ini"], m["fin"]) for m in menc]
        for m in menc_extra:
            if (m["tipo"], norm(m["nombre"]), m["ini"]) in ya:
                continue
            if any(a <= m["ini"] < b for a, b in cubiertos) and m["tipo"] in ("Laguna", "Glaciar", "Nevado", "CentroPoblado"):
                continue
            menc.append(m)
    base = {"paper_id": c["paper_id"], "chunk_id": c["chunk_id"], "section": c.get("section_type", ""),
            "page_start": c.get("page_start"), "page_end": c.get("page_end"), "source": c.get("fuente", "")}
    trip = []
    foco = [m for m in menciones_reglas(c.get("title", "") or "") if m["tipo"] == "Laguna"][:3]
    for m in foco:
        m["metodo"] = "reglas"
    ors = oraciones(texto)
    for k, o in enumerate(ors):
        dentro = [m for m in menc if o[0] <= m["ini"] < o[1]]
        for m in dentro:
            m["oracion"] = k
        if dentro or re.search(EVENTOS, o[2], re.I) or (foco and re.search(r"volumen|profundidad|espejo|[áa]rea|altitud|cota|s\.\s?n\.\s?m|msnm|riesgo|peligro|dique|t[uú]nel|sif[oó]n|alerta temprana|aguas abajo|se form|apareci|estado|mantenimiento", o[2], re.I)):
            trip += relaciones_oracion(o, dentro, base, foco)
        lag = list({m["nombre"]: m for m in dentro if m["tipo"] == "Laguna"}.values())
        if lag:
            foco = lag[:3]
    menciones = [{**base, "tipo": m["tipo"], "nombre": m["nombre"], "metodo": m["metodo"], "oracion": m.get("oracion", -1),
                  "ini": m["ini"], "fin": m.get("fin", m["ini"]),
                  "texto": texto[m["ini"]:m.get("fin", m["ini"])][:80]} for m in menc]
    return menciones, trip


# ======================================================================
# 6. LLM (Gemini) sobre una muestra
# ======================================================================
PROMPT_LLM = """Eres un analista experto en glaciares y peligros glaciares del Perú. Del siguiente fragmento de un documento técnico
extrae SOLO hechos dichos explícitamente en el texto. No inventes ni completes con conocimiento propio.

Tipos de entidad permitidos: {tipos}
Predicados permitidos (sujeto → objeto):
{predicados}

Reglas:
- Nombres sin prefijo: "Palcacocha" (no "laguna Palcacocha"), "Blanca" para la Cordillera Blanca, "Huaraz".
- Para valores (riesgo, volumen, área, altitud, año) usa objeto literal con unidad, p. ej. "17,4 hm3", "Muy alto", "1941", y tipo_objeto "Literal".
- Un evento se nombra con tipo, lugar y año: "Aluvión Palcacocha 1941".
- "evidencia" debe ser una frase copiada LITERALMENTE del fragmento que respalde la relación.
- Si no hay hechos del dominio, devuelve listas vacías.

Responde solo con JSON: {{"entidades":[{{"nombre":"","tipo":""}}],"relaciones":[{{"sujeto":"","tipo_sujeto":"","predicado":"","objeto":"","tipo_objeto":"","evidencia":""}}]}}

Documento: {titulo} ({anio}) · sección: {seccion}
Fragmento:
\"\"\"{texto}\"\"\""""


def prompt_llm(c):
    preds = "\n".join(f"- {k}: {v}" for k, v in PREDICADOS.items())
    return PROMPT_LLM.format(tipos=", ".join(TIPOS), predicados=preds, titulo=c.get("title", ""), anio=c.get("year", ""),
                             seccion=c.get("section_type", ""), texto=c["text"][:3500])


def modelos_disponibles(client, preferidos=()):
    """Modelos 'flash' que la clave puede usar, del más preferido al menos (Google retira nombres viejos)."""
    try:
        lista = [(m.name or "").replace("models/", "") for m in client.models.list()
                 if not getattr(m, "supported_actions", None) or "generateContent" in m.supported_actions]
    except Exception:
        return list(preferidos)
    def version(n):
        v = re.search(r"gemini-(\d+(?:\.\d+)?)", n)
        return float(v.group(1)) if v else 0.0
    flash = [n for n in lista if "flash" in n and not re.search(r"image|tts|audio|live|thinking|embedding|exp", n)]
    estables = sorted([n for n in flash if "preview" not in n], key=lambda n: (-version(n), "lite" in n, n))
    previas = sorted([n for n in flash if "preview" in n], key=lambda n: (-version(n), "lite" in n, n))
    orden = [p for p in preferidos if p in lista] + [n for n in ("gemini-flash-latest", "gemini-flash-lite-latest") if n in lista]
    return list(dict.fromkeys(orden + estables + previas)) or list(preferidos)


class ClienteGemini:
    def __init__(self, api_key, modelos=("gemini-flash-latest",), pausa=6.0):
        from google import genai
        from google.genai import types
        self.client, self.types = genai.Client(api_key=api_key), types
        self.modelos = modelos_disponibles(self.client, modelos)
        self.i_modelo, self.pausa = 0, pausa
        self._ultimo = 0.0

    @property
    def modelo(self):
        return self.modelos[self.i_modelo]

    def generar(self, prompt):
        espera = self.pausa - (time.time() - self._ultimo)
        if espera > 0:
            time.sleep(espera)
        cfg = dict(response_mime_type="application/json", temperature=0)
        try:
            cfg["thinking_config"] = self.types.ThinkingConfig(thinking_budget=0)
            config = self.types.GenerateContentConfig(**cfg)
        except Exception:
            cfg.pop("thinking_config", None)
            config = self.types.GenerateContentConfig(**cfg)
        self._ultimo = time.time()
        r = self.client.models.generate_content(model=self.modelo, contents=prompt, config=config)
        return r.text or ""


def _json_llm(txt):
    txt = (txt or "").strip()
    txt = re.sub(r"^```(?:json)?|```$", "", txt).strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", txt, re.S)
        return json.loads(m.group(0)) if m else {}


def evidencia_en_texto(ev, texto):
    e, t = norm(ev), norm(texto)
    if not e:
        return False
    if e in t:
        return True
    pal = [p for p in re.findall(r"\w+", e) if len(p) > 2]
    return bool(pal) and sum(p in t for p in pal) / len(pal) >= 0.85


def validar_llm(datos, c):
    """Convierte la respuesta del LLM en tripletas con el mismo formato que las reglas."""
    base = {"paper_id": c["paper_id"], "chunk_id": c["chunk_id"], "section": c.get("section_type", ""),
            "page_start": c.get("page_start"), "page_end": c.get("page_end"), "source": c.get("fuente", "")}
    trip, menc, descartes = [], [], Counter()
    for e in datos.get("entidades", []) or []:
        tipo = e.get("tipo")
        if tipo in TIPOS and e.get("nombre"):
            n = canon_nombre(tipo, str(e["nombre"])) or str(e["nombre"]).strip()
            menc.append({**base, "tipo": tipo, "nombre": n, "metodo": "llm", "texto": str(e["nombre"])[:80]})
    for r in datos.get("relaciones", []) or []:
        p, ts, to = r.get("predicado"), r.get("tipo_sujeto"), r.get("tipo_objeto") or "Literal"
        if p not in PREDICADOS:
            descartes["predicado_no_permitido"] += 1; continue
        if ts not in TIPOS or (to not in TIPOS and to != "Literal"):
            descartes["tipo_no_permitido"] += 1; continue
        s = canon_nombre(ts, str(r.get("sujeto", ""))) or str(r.get("sujeto", "")).strip()
        o = str(r.get("objeto", "")).strip()
        o = (canon_nombre(to, o) or o) if to != "Literal" else o
        if not s or not o:
            descartes["vacio"] += 1; continue
        ok = evidencia_en_texto(r.get("evidencia", ""), c["text"])
        if not ok:
            descartes["evidencia_no_verificada"] += 1
        m = re.match(r"^\s*" + NUM, o) if to == "Literal" else None
        trip.append({**base, "subject": s, "subject_type": ts, "predicate": p, "object": o, "object_type": to,
                     "value": numero(m.group("num")) if m else None, "unit": (_unidad(o, p) if m else ""),
                     "evidence": str(r.get("evidencia", ""))[:600], "confidence": 0.85 if ok else 0.5,
                     "method": "llm" if ok else "llm_sin_verificar"})
    return menc, trip, descartes


def extraer_llm(cliente, chunks, cache_ruta, max_errores=3, log=print):
    """Llama al LLM fragmento por fragmento, guardando cada respuesta (reanudable)."""
    hechos = {}
    if os.path.exists(cache_ruta):
        for r in leer_jsonl(cache_ruta):
            hechos[r["chunk_id"]] = r
    pendientes = [c for c in chunks if c["chunk_id"] not in hechos]
    log(f"LLM ({cliente.modelo}): {len(hechos)} en caché, {len(pendientes)} por procesar "
        f"(~{len(pendientes) * cliente.pausa / 60:.0f} min)")
    errores_seguidos = 0
    with open(cache_ruta, "a", encoding="utf-8") as fh:
        for k, c in enumerate(pendientes, start=1):
            intento, espera = 0, 30
            while True:
                try:
                    txt = cliente.generar(prompt_llm(c))
                    datos = _json_llm(txt)
                    reg = {"chunk_id": c["chunk_id"], "modelo": cliente.modelo, "respuesta": datos, "fecha": ahora()}
                    fh.write(json.dumps(reg, ensure_ascii=False) + "\n"); fh.flush()
                    hechos[c["chunk_id"]] = reg
                    errores_seguidos = 0
                    break
                except Exception as e:
                    msg = str(e)
                    if ("404" in msg or "not found" in msg.lower() or "no longer available" in msg.lower()) \
                            and cliente.i_modelo + 1 < len(cliente.modelos):
                        cliente.i_modelo += 1
                        log(f"   modelo no disponible; se usa {cliente.modelo}")
                        continue
                    saturado = any(x in msg for x in ("503", "UNAVAILABLE", "500", "INTERNAL", "overloaded", "high demand"))
                    if saturado and intento < 3:
                        intento += 1
                        log(f"   {cliente.modelo} saturado (temporal); reintento en {espera} s")
                        time.sleep(espera); espera = min(espera * 2, 120)
                        continue
                    if saturado and cliente.i_modelo + 1 < len(cliente.modelos):
                        cliente.i_modelo += 1          # ese modelo sigue saturado: se prueba otro modelo vigente
                        intento, espera = 0, 30
                        log(f"   se cambia al modelo {cliente.modelo}")
                        continue
                    if ("429" in msg or "RESOURCE_EXHAUSTED" in msg or "quota" in msg.lower()) and intento < 3:
                        intento += 1
                        log(f"   límite de uso por minuto; espera {espera} s")
                        time.sleep(espera); espera = min(espera * 2, 240)
                        continue
                    if isinstance(e, json.JSONDecodeError) or "Expecting" in msg or "validate JSON" in msg or "json_validate" in msg:
                        reg = {"chunk_id": c["chunk_id"], "modelo": cliente.modelo, "respuesta": {}, "error": "json", "fecha": ahora()}
                        fh.write(json.dumps(reg, ensure_ascii=False) + "\n"); fh.flush()
                        hechos[c["chunk_id"]] = reg
                        break
                    errores_seguidos += 1
                    log(f"   ⚠️ {c['chunk_id']}: {msg[:160]}")
                    break
            if errores_seguidos >= max_errores and cliente.i_modelo + 1 < len(cliente.modelos) and "404" in msg:
                cliente.i_modelo += 1
                errores_seguidos = 0
                log(f"   se prueba con {cliente.modelo}")
                continue
            if errores_seguidos >= max_errores:
                log("⛔ Se detiene la llamada al LLM (cuota diaria agotada o Gemini saturado). "
                    "Lo procesado queda guardado: volver a ejecutar más tarde y continúa donde quedó.")
                break
            if k % 10 == 0 or k == len(pendientes):
                log(f"   LLM {k}/{len(pendientes)}")
    return hechos


def muestra_llm(chunks, n=200, max_por_doc=2, docs_demo=(), semilla=7):
    """Fragmentos más informativos, repartidos entre documentos (+ todos los del documento de demo)."""
    df = pd.DataFrame(chunks)
    secciones = {"resultados", "conclusiones", "desarrollo", "area_estudio", "antecedentes", "cuerpo",
                 "introduccion", "resumen", "objetivos", "metodologia"}
    d = df[df.section_type.isin(secciones) & (df.n_chars >= 300)]
    d = d.sample(frac=1, random_state=semilla).sort_values("prioridad", ascending=False, kind="stable")
    d = d.groupby("paper_id", sort=False).head(max_por_doc).head(n)
    demo = df[df.paper_id.isin(docs_demo) & (df.prioridad >= 2)].head(12)
    ids = list(dict.fromkeys(list(demo.chunk_id) + list(d.chunk_id)))
    por_id = {c["chunk_id"]: c for c in chunks}
    return [por_id[i] for i in ids]


# ======================================================================
# 7. Consolidación, entidades y salidas
# ======================================================================
COLS_TRIP = ["subject", "subject_type", "predicate", "object", "object_type", "paper_id", "section", "evidence",
             "method", "source", "subject_id", "object_id", "chunk_id", "page_start", "page_end", "value", "unit",
             "year", "value_std", "unit_std", "verb", "confidence"]


def preparar_tripletas(trip):
    df = pd.DataFrame(trip)
    if not len(df):
        return pd.DataFrame(columns=COLS_TRIP)
    df["subject_id"] = [clave(t, n) for t, n in zip(df.subject_type, df.subject)]
    df["object_id"] = [clave(t, n) if t != "Literal" else "" for t, n in zip(df.object_type, df.object)]
    for c in COLS_TRIP:
        if c not in df.columns:
            df[c] = ""
    std = [normalizar_valor(p, v, u) for p, v, u in zip(df.predicate, df.value, df.unit)]
    df["value_std"] = [x[0] for x in std]
    df["unit_std"] = [x[1] for x in std]
    df = df[COLS_TRIP].drop_duplicates(["subject_id", "predicate", "object", "chunk_id", "method"])
    return df.reset_index(drop=True)


def consolidar(df):
    """Una fila por hecho (sujeto, predicado, objeto) con cuántas veces y en cuántos documentos aparece."""
    if not len(df):
        return df
    g = df.sort_values("confidence", ascending=False).groupby(["subject_id", "predicate", "object"], sort=False)
    out = g.agg(subject=("subject", "first"), subject_type=("subject_type", "first"), object_type=("object_type", "first"),
                object_id=("object_id", "first"), n_evidencias=("chunk_id", "size"), n_docs=("paper_id", "nunique"),
                confianza_max=("confidence", "max"), metodos=("method", lambda x: ";".join(sorted(set(x)))),
                paper_id=("paper_id", "first"), chunk_id=("chunk_id", "first"), page_start=("page_start", "first"),
                page_end=("page_end", "first"), evidencia=("evidence", "first"), year=("year", "first"),
                value=("value", "first"), unit=("unit", "first"), value_std=("value_std", "first"),
                unit_std=("unit_std", "first"), verb=("verb", "first"),
                documentos=("paper_id", lambda x: ";".join(list(dict.fromkeys(x))[:10]))).reset_index()
    # confianza combinada: más documentos y más MÉTODOS distintos que coinciden → más confiable (votación)
    out["n_metodos"] = out.metodos.map(lambda m: len({FAMILIA.get(x, x) for x in m.split(";") if x}))
    out["confianza"] = (out.confianza_max + 0.05 * (out.n_docs.clip(upper=5) - 1)
                        + 0.07 * (out.n_metodos.clip(upper=4) - 1)).clip(upper=0.99).round(2)
    return out.sort_values(["n_docs", "n_evidencias"], ascending=False).reset_index(drop=True)


def tabla_entidades(menciones):
    m = pd.DataFrame(menciones)
    if not len(m):
        return pd.DataFrame(columns=["entidad_id", "tipo", "nombre", "n_menciones", "n_docs", "metodos", "variantes"])
    m["entidad_id"] = [clave(t, n) for t, n in zip(m.tipo, m.nombre)]
    e = m.groupby("entidad_id").agg(tipo=("tipo", "first"), nombre=("nombre", lambda x: Counter(x).most_common(1)[0][0]),
                                    n_menciones=("chunk_id", "size"), n_docs=("paper_id", "nunique"),
                                    metodos=("metodo", lambda x: ";".join(sorted(set(x)))),
                                    variantes=("texto", lambda x: " | ".join(list(dict.fromkeys(x))[:6]))).reset_index()
    return e.sort_values(["tipo", "n_docs"], ascending=[True, False]).reset_index(drop=True)


def tripletas_documento(menciones, metadatos_papers=None, min_menciones=1):
    """Documento —mencionaA→ entidad (une cada documento con lo que trata; base del GraphRAG)."""
    m = pd.DataFrame(menciones)
    if not len(m):
        return pd.DataFrame(columns=COLS_TRIP)
    m = m[~m.tipo.isin(["MetodoMonitoreo", "ObraSeguridad", "ServicioEcosistemico"])]
    g = m.groupby(["paper_id", "tipo", "nombre"]).agg(n=("chunk_id", "size"), chunk_id=("chunk_id", "first"),
                                                     page_start=("page_start", "first"), page_end=("page_end", "first"),
                                                     source=("source", "first")).reset_index()
    g = g[g.n >= min_menciones]
    out = pd.DataFrame({"subject": g.paper_id, "subject_type": "Documento", "predicate": "mencionaA", "object": g.nombre,
                        "object_type": g.tipo, "paper_id": g.paper_id, "section": "", "evidence": "",
                        "method": "agregado", "source": g.source, "subject_id": g.paper_id,
                        "object_id": [clave(t, n) for t, n in zip(g.tipo, g.nombre)], "chunk_id": g.chunk_id,
                        "page_start": g.page_start, "page_end": g.page_end, "value": g.n, "unit": "menciones",
                        "confidence": (0.5 + 0.1 * g.n.clip(upper=5)).round(2)})
    met = pd.DataFrame(menciones)
    met = met[met.tipo == "MetodoMonitoreo"].groupby(["paper_id", "nombre"]).agg(
        n=("chunk_id", "size"), chunk_id=("chunk_id", "first"), source=("source", "first")).reset_index()
    out2 = pd.DataFrame({"subject": met.paper_id, "subject_type": "Documento", "predicate": "usaMetodo",
                         "object": met.nombre, "object_type": "MetodoMonitoreo", "paper_id": met.paper_id, "section": "",
                         "evidence": "", "method": "agregado", "source": met.source, "subject_id": met.paper_id,
                         "object_id": [clave("MetodoMonitoreo", n) for n in met.nombre], "chunk_id": met.chunk_id,
                         "page_start": None, "page_end": None, "value": met.n, "unit": "menciones",
                         "confidence": (0.5 + 0.1 * met.n.clip(upper=5)).round(2)})
    return pd.concat([out, out2], ignore_index=True).reindex(columns=COLS_TRIP)


def unir_con_metadatos(df_contenido, ruta_meta):
    """tripletas_todas.csv: metadatos (Persona 1) + contenido (Persona 3), mismo formato."""
    partes = [df_contenido]
    if ruta_meta and os.path.exists(ruta_meta):
        meta = pd.read_csv(ruta_meta, dtype=str).fillna("")
        meta["subject_id"] = meta["subject"]
        meta["object_id"] = ""
        meta["confidence"] = 1.0
        for c in COLS_TRIP:
            if c not in meta.columns:
                meta[c] = ""
        partes.insert(0, meta[COLS_TRIP])
    return pd.concat(partes, ignore_index=True)


# Hechos de referencia (conocimiento base del dominio, se marcan como tales)
HECHOS_REFERENCIA = [
    ("Blanca", "Cordillera", "Áncash"), ("Negra", "Cordillera", "Áncash"), ("Huallanca", "Cordillera", "Áncash"),
("Huayhuash", "Cordillera", "Áncash"), ("Huayhuash", "Cordillera", "Huánuco"), ("Huayhuash", "Cordillera", "Lima"), ("Raura", "Cordillera", "Lima"), ("Huaytapallana", "Cordillera", "Junín"),
    ("Central", "Cordillera", "Lima"), ("Vilcanota", "Cordillera", "Cusco"), ("Urubamba", "Cordillera", "Cusco"),
    ("Vilcabamba", "Cordillera", "Cusco"), ("Carabaya", "Cordillera", "Puno"), ("Apolobamba", "Cordillera", "Puno"),
    ("Ampato", "Cordillera", "Arequipa"), ("Chila", "Cordillera", "Arequipa"), ("La Raya", "Cordillera", "Puno"),
    ("Huagoruncho", "Cordillera", "Pasco"), ("La Viuda", "Cordillera", "Lima"), ("Chonta", "Cordillera", "Huancavelica"),
    ("Huanzo", "Cordillera", "Apurímac"), ("Volcánica", "Cordillera", "Arequipa"), ("Barroso", "Cordillera", "Tacna"),
    ("Huaraz", "Provincia", "Áncash"), ("Carhuaz", "Provincia", "Áncash"), ("Yungay", "Provincia", "Áncash"),
    ("Huaylas", "Provincia", "Áncash"), ("Recuay", "Provincia", "Áncash"), ("Huari", "Provincia", "Áncash"),
]


def tripletas_referencia():
    filas = [{"subject": s, "subject_type": t, "predicate": "ubicadaEn", "object": d, "object_type": "Departamento",
              "paper_id": "", "section": "", "evidence": "conocimiento de referencia (geografía del Perú)",
              "method": "referencia", "source": "referencia", "chunk_id": "", "page_start": None, "page_end": None,
              "value": None, "unit": "", "confidence": 0.95} for s, t, d in HECHOS_REFERENCIA]
    return preparar_tripletas(filas)


# ======================================================================
# 8. Consultas de demostración (preguntas de competencia sobre las tripletas)
# ======================================================================
def _ids(cons, pred, obj_tipo=None, obj_nombre=None, suj_tipo=None):
    d = cons[cons.predicate == pred]
    if obj_tipo:
        d = d[d.object_type == obj_tipo]
    if obj_nombre:
        d = d[d.object.map(norm) == norm(obj_nombre)]
    if suj_tipo:
        d = d[d.subject_type == suj_tipo]
    return d


def lagunas_en_departamento(cons, departamento):
    """Multi-salto: laguna → (cordillera | provincia | cuenca | departamento) → departamento."""
    ubic = cons[cons.predicate == "ubicadaEn"]
    dep = norm(departamento)
    lugares = set(ubic[(ubic.object_type == "Departamento") & (ubic.object.map(norm) == dep)].subject_id)
    lag = ubic[(ubic.subject_type == "Laguna") & ((ubic.object_id.isin(lugares)) |
                                                  ((ubic.object_type == "Departamento") & (ubic.object.map(norm) == dep)))]
    # si el texto dice explícitamente que la laguna está en OTRO departamento (y nunca en este), se descarta
    deps = ubic[(ubic.subject_type == "Laguna") & (ubic.object_type == "Departamento")]
    explicitos = deps.groupby("subject_id").object.agg(lambda x: {norm(v) for v in x})
    otros = {k for k, v in explicitos.items() if dep not in v}
    return lag[~lag.subject_id.isin(otros)]


def consulta_riesgo(cons, departamento="Áncash", nivel="Muy alto"):
    lag = lagunas_en_departamento(cons, departamento)
    r = cons[(cons.predicate == "tieneNivelRiesgo") & (cons.object.map(norm) == norm(nivel)) &
             cons.subject_id.isin(lag.subject_id)]
    out = r.merge(lag[["subject_id", "object"]].rename(columns={"object": "via"}), on="subject_id")
    return (out.groupby(["subject"]).agg(via=("via", lambda x: ", ".join(sorted(set(x)))),
                                         n_docs=("n_docs", "max"), documento=("paper_id", "first"),
                                         pagina=("page_start", "first"), evidencia=("evidencia", "first"))
            .reset_index().rename(columns={"subject": "laguna"}).sort_values("n_docs", ascending=False))


def consulta_poblados_amenazados(cons, laguna=None):
    d = cons[cons.predicate.isin(["amenazaA", "afectoA"])]
    if laguna:
        d = d[d.subject.map(norm).str.contains(norm(laguna))]
    return d[["subject", "predicate", "object", "n_docs", "paper_id", "page_start", "evidencia"]]


def consulta_entidad(cons, nombre):
    k = norm(nombre)
    d = cons[(cons.subject.map(norm) == k) | (cons.object.map(norm) == k)]
    return d[["subject", "predicate", "object", "n_docs", "metodos", "paper_id", "page_start", "evidencia"]]


def dibujar_vecindario(cons, nombre, max_aristas=40, ruta=None):
    import networkx as nx
    import matplotlib.pyplot as plt
    d = consulta_entidad(cons, nombre).sort_values("n_docs", ascending=False).head(max_aristas)
    G = nx.DiGraph()
    for r in d.itertuples():
        G.add_edge(r.subject, r.object, label=r.predicate)
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=3, k=0.9)
    colores = ["#d62728" if norm(n) == norm(nombre) else "#1f77b4" for n in G.nodes]
    nx.draw_networkx(G, pos, node_color=colores, node_size=900, font_size=8, alpha=0.9, arrows=True)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=7)
    plt.title(f"Vecindario de '{nombre}' en el grafo de conocimiento (tripletas extraídas)")
    plt.axis("off")
    if ruta:
        plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.show()


# ======================================================================
# 9. Reporte
# ======================================================================
def reporte_pln(n_chunks, n_docs, df_trip, cons, entidades, info):
    ok, warn = "✅", "⚠️"
    chk = []
    docs_con = df_trip[df_trip.predicate != "mencionaA"].paper_id.nunique() if len(df_trip) else 0
    chk.append((ok if docs_con >= 0.6 * n_docs else warn, f"Documentos con al menos una relación extraída: {docs_con}/{n_docs}"))
    chk.append((ok if len(entidades) >= 200 else warn, f"Entidades distintas: {len(entidades)}"))
    rel = cons[~cons.predicate.isin(["mencionaA", "usaMetodo"])] if len(cons) else cons
    chk.append((ok if len(rel) >= 500 else warn, f"Hechos distintos (sin contar menciones): {len(rel)}"))
    multi = int((rel.n_docs >= 2).sum()) if len(rel) else 0
    chk.append((ok, f"Hechos confirmados por 2 o más documentos: {multi}"))
    if info.get("llm_chunks"):
        chk.append((ok if info.get("llm_verificadas", 0) >= 0.7 * max(info.get("llm_total", 1), 1) else warn,
                    f"Tripletas del LLM con evidencia verificada en el texto: {info.get('llm_verificadas', 0)}/{info.get('llm_total', 0)}"))
    lineas = ["# Reporte de extracción de entidades y relaciones (Persona 3 · Glaciares)", "",
              f"Generado: {ahora()} · versión {VERSION}", "",
              f"Fragmentos procesados: reglas {n_chunks} · GLiNER {info.get('gliner_chunks', 0)} · LLM {info.get('llm_chunks', 0)} "
              f"({info.get('llm_modelo', '-')})", "", "| Estado | Verificación |", "|---|---|"] + [f"| {e} | {m} |" for e, m in chk]
    if len(entidades):
        lineas += ["", "## Entidades por tipo", ""] + [f"- {k}: {v}" for k, v in entidades.tipo.value_counts().items()]
        for tipo in ("Laguna", "Glaciar", "Nevado", "CentroPoblado", "Cordillera"):
            top = entidades[entidades.tipo == tipo].sort_values("n_docs", ascending=False).head(10)
            if len(top):
                lineas += ["", f"Más mencionadas ({tipo}): " + ", ".join(f"{r.nombre} ({r.n_docs} docs)" for r in top.itertuples())]
    if len(df_trip):
        lineas += ["", "## Tripletas por predicado", ""] + \
                  [f"- {k}: {v}" for k, v in df_trip.predicate.value_counts().items()]
        lineas += ["", "## Tripletas por método", ""] + [f"- {k}: {v}" for k, v in df_trip["method"].value_counts().items()]
    if info.get("comparacion"):
        lineas += ["", "## Reglas vs LLM (mismos fragmentos)", ""] + [f"- {k}: {v}" for k, v in info["comparacion"].items()]
    return chk, "\n".join(lineas)


README_TRIPLETAS = """# Entidades y tripletas del corpus Glaciares (Persona 3)

Entrada: `texto_glaciares/chunks.jsonl` (Persona 2). Cada hecho guarda su procedencia:
`paper_id` (handle del repositorio), `chunk_id`, páginas y la oración de evidencia.

| Archivo | Contenido |
|---|---|
| `entidades.csv` | Una fila por entidad (laguna, glaciar, nevado, cordillera, cuenca, centro poblado, institución, método, obra…) con nº de documentos y variantes |
| `menciones.csv` | Cada mención de una entidad en un fragmento |
| `tripletas_contenido.csv` | Una fila por evidencia: sujeto, predicado, objeto, documento, páginas, oración, método (reglas, gliner+reglas, llm), confianza |
| `tripletas_consolidadas.csv` | Una fila por hecho único con nº de evidencias y de documentos (lo que se carga al grafo) |
| `tripletas_todas.csv` | Metadatos (Persona 1) + contenido, en el mismo formato |
| `tripletas_descartadas.csv` | Hechos rechazados por las reglas de sentido común, con el motivo |
| `alias_entidades.csv` | Variantes unificadas (p. ej. Palcaccocha → Palcacocha) |
| `reporte_pln.md` | Verificaciones, estadísticas y cobertura de las 10 preguntas de competencia |
| `cache_gliner.jsonl`, `cache_llm.jsonl` | Caché para continuar sin repetir (no se suben a GitHub si pesan mucho) |

## Tipos de entidad
`Laguna, Glaciar, Nevado, Cordillera, Cuenca, Rio, Quebrada, Departamento, Provincia, Distrito, CentroPoblado,
Evento, Institucion, MetodoMonitoreo, ObraSeguridad` (+ `Documento` de la Persona 1).

## Predicados
`ubicadaEn, alimentadaPor, tieneNivelRiesgo, amenazaA, ocurrioEn, afectoA, ocurrioEnAnio, tieneObraSeguridad,
monitoreadoCon, intervieneEn, tieneVolumen, tieneArea, tieneAltitud, tieneProfundidadMax, tieneSuperficieGlaciar,
mencionaA (Documento→entidad), usaMetodo (Documento→método)` + los de metadatos de la Persona 1.

## Métodos de extracción
- `reglas`: patrones y diccionarios del dominio sobre cada oración (todo el corpus).
- `tabla`: fichas y tablas de inventarios (área, volumen, altitud, ubicación).
- `diccionario`: nombres conocidos (vistos en ≥2 documentos) encontrados sueltos en el texto.
- `spacy`: análisis gramatical sujeto → verbo → objeto; si el verbo no está en la ontología queda `relacionadoCon` + `verb`.
- `gliner+reglas`: GLiNER multilingüe detecta nombres que las reglas no ven; la relación la decide la regla.
- `llm`: Gemini sobre una muestra; solo se aceptan predicados de la lista y se verifica que la evidencia esté en el texto
  (`llm_sin_verificar` si no se encontró).
- `referencia`: pocos hechos geográficos fijos (cordillera → departamento) para las consultas de varios saltos.
- `agregado`: Documento —mencionaA→ entidad.

`confianza` (0–1) sube cuando el hecho aparece en varios documentos y cuando lo encuentran varias técnicas distintas
(`n_metodos`, votación). `value_std` y `unit_std` traen los valores en unidades comunes (m³, km², m).
"""


# ======================================================================
# 10. v2 — Unidades, sentido común y votación
# ======================================================================
# familias de métodos para la votación (gliner+reglas cuenta como gliner)
FAMILIA = {"reglas": "reglas", "gliner+reglas": "gliner", "diccionario": "diccionario", "spacy": "spacy",
           "tabla": "tabla", "llm": "llm", "llm_sin_verificar": "llm", "agregado": "agregado", "referencia": "referencia"}


def normalizar_valor(pred, valor, unidad):
    """Convierte a una unidad común: volumen → m3, área → km2, altitud y profundidad → m."""
    try:
        v = float(valor)
    except (TypeError, ValueError):
        return None, ""
    u = norm(str(unidad or "")).replace(" ", "")
    if pred == "tieneVolumen":
        if u in ("hm3", "mmc", "mmm3") or "millones" in u:
            return v * 1e6, "m3"
        return v, "m3"
    if pred in ("tieneArea", "tieneSuperficieGlaciar"):
        if u in ("m2",):
            return v / 1e6, "km2"
        if u == "ha":
            return v / 100, "km2"
        return v, "km2"
    if pred in ("tieneAltitud", "tieneProfundidadMax"):
        return v, "m"
    if pred in ("causoVictimas",):
        return v, "personas"
    if pred in ("tienePoblacion",):
        return v, "habitantes"
    if pred in ("ocurrioEnAnio", "formadaEnAnio"):
        return v, "año"
    return v, unidad or ""


LUGARES = {"Cordillera", "Cuenca", "Quebrada", "Rio", "Departamento", "Provincia", "Distrito"}
GEO = {"Laguna", "Glaciar", "Nevado", "Bofedal", "Cuenca", "Rio", "Quebrada", "Cordillera", "Distrito", "Provincia",
       "Departamento", "CentroPoblado"}
# (tipos de sujeto, tipos de objeto) permitidos por predicado — "Literal" para valores
DOMINIO = {
    "ubicadaEn": (GEO | {"Ecosistema"}, LUGARES),
    "alimentadaPor": ({"Laguna", "Rio", "Bofedal", "Cuenca"}, {"Glaciar", "Nevado", "Laguna"}),
    "tieneNivelRiesgo": ({"Laguna", "Glaciar", "Nevado", "Quebrada", "CentroPoblado", "Cuenca"}, {"Literal"}),
    "amenazaA": ({"Laguna", "Glaciar", "Nevado", "Quebrada", "Evento"}, {"CentroPoblado", "Distrito", "Provincia"}),
    "ocurrioEn": ({"Evento"}, {"Laguna", "Glaciar", "Nevado", "Quebrada", "Cuenca", "Rio", "CentroPoblado"}),
    "afectoA": ({"Evento"}, {"CentroPoblado", "Distrito", "Provincia", "Cuenca", "Rio", "Quebrada"}),
    "ocurrioEnAnio": ({"Evento"}, {"Literal"}),
    "causoVictimas": ({"Evento"}, {"Literal"}),
    "tieneObraSeguridad": ({"Laguna"}, {"ObraSeguridad", "Literal"}),
    "estadoObra": ({"Laguna"}, {"Literal"}),
    "monitoreadoCon": ({"Laguna", "Glaciar", "Nevado", "Bofedal", "Cuenca", "Cordillera"}, {"MetodoMonitoreo", "Literal"}),
    "intervieneEn": ({"Institucion"}, {"Laguna", "Glaciar", "Nevado", "Bofedal", "Cuenca", "Cordillera", "Quebrada"}),
    "tieneVolumen": ({"Laguna"}, {"Literal"}),
    "tieneArea": ({"Laguna", "Glaciar", "Nevado", "Bofedal"}, {"Literal"}),
    "tieneAltitud": ({"Laguna", "Glaciar", "Nevado", "Bofedal", "CentroPoblado"}, {"Literal"}),
    "tieneProfundidadMax": ({"Laguna"}, {"Literal"}),
    "tieneSuperficieGlaciar": ({"Cordillera", "Nevado", "Glaciar", "Cuenca"}, {"Literal"}),
    "formadaEnAnio": ({"Laguna"}, {"Literal"}),
    "tienePoblacion": ({"CentroPoblado", "Distrito", "Provincia"}, {"Literal"}),
    "tieneEcosistema": (GEO | {"Ecosistema"}, {"Ecosistema"}),
    "presentaFenomeno": (GEO, {"Fenomeno"}),
    "albergaEspecie": (GEO | {"Ecosistema"}, {"Especie"}),
    "proveeServicio": ({"Bofedal", "Ecosistema", "Glaciar", "Nevado", "Laguna", "Cuenca", "Rio"}, {"ServicioEcosistemico"}),
    "abasteceA": ({"Laguna", "Rio", "Glaciar", "Nevado", "Bofedal", "Cuenca"}, {"CentroPoblado", "Distrito", "Provincia"}),
    "drenaHacia": ({"Laguna", "Rio", "Quebrada", "Bofedal"}, {"Rio", "Quebrada", "Laguna", "Cuenca"}),
}
# rangos razonables (en unidades normalizadas)
RANGOS = {"tieneVolumen": (1e2, 2e9), "tieneArea": (1e-4, 300), "tieneAltitud": (1500, 6800),
          "tieneProfundidadMax": (0.3, 400), "tieneSuperficieGlaciar": (0.01, 3000), "ocurrioEnAnio": (1500, 2026),
          "formadaEnAnio": (1500, 2026), "causoVictimas": (1, 100000), "tienePoblacion": (10, 15e6)}


def validar_tripletas(df):
    """Reglas de sentido común: tipos permitidos por predicado, valores dentro de rangos y sin bucles.
    Devuelve (válidas, descartadas con motivo)."""
    if not len(df):
        return df, df.assign(motivo=[])
    motivos = []
    for r in df.itertuples():
        m = ""
        regla = DOMINIO.get(r.predicate)
        if r.subject_type == "Documento" or r.predicate in ("mencionaA", "usaMetodo"):
            m = ""
        elif r.predicate == "relacionadoCon":
            m = "mismo_sujeto_objeto" if r.subject_id == r.object_id else ""
        elif regla:
            if r.subject_type not in regla[0]:
                m = f"tipo_sujeto ({r.subject_type})"
            elif r.object_type not in regla[1]:
                m = f"tipo_objeto ({r.object_type})"
        if not m and r.object_type != "Literal" and r.subject_id == r.object_id:
            m = "mismo_sujeto_objeto"
        if not m and r.predicate in RANGOS:
            lo, hi = RANGOS[r.predicate]
            try:
                v = float(r.value_std)
                if not (lo <= v <= hi):
                    m = f"fuera_de_rango ({v:g} {r.unit_std})"
            except (TypeError, ValueError):
                if r.predicate in ("ocurrioEnAnio", "formadaEnAnio"):
                    m = "anio_invalido"
        motivos.append(m)
    df = df.assign(motivo=motivos)
    return df[df.motivo == ""].drop(columns="motivo").reset_index(drop=True), df[df.motivo != ""].reset_index(drop=True)


# ======================================================================
# 11. v2 — Tablas de inventarios (Persona 2: tables.jsonl)
# ======================================================================
import html as _html

COL_NOMBRE = re.compile(r"\b(laguna|lagunas|nombre|glaciar|nevado|lake|glacier|bofedal)\b", re.I)
COL_MEDIDA = [
    ("tieneArea", re.compile(r"[áa]rea|superficie|espejo|area", re.I)),
    ("tieneVolumen", re.compile(r"volumen|volume", re.I)),
    ("tieneAltitud", re.compile(r"altitud|cota|altura|elevaci|m\.?\s?s\.?\s?n\.?\s?m|msnm|altitude", re.I)),
    ("tieneProfundidadMax", re.compile(r"profundidad\s*m[aá]x|prof\.?\s*m[aá]x|max(imum)?\.? depth", re.I)),
]
COL_LUGAR = [("Distrito", re.compile(r"^distrito", re.I)), ("Provincia", re.compile(r"^provincia", re.I)),
             ("Departamento", re.compile(r"^(departamento|regi[oó]n)", re.I)),
             ("Cuenca", re.compile(r"^(sub|micro)?cuenca|unidad hidrogr", re.I)),
             ("Cordillera", re.compile(r"^cordillera", re.I))]
UNIDAD = re.compile(r"(km2|km²|m2|m²|\bha\b|hm3|hm³|MMC|MMm3|m3|m³|m\.?\s?s\.?\s?n\.?\s?m\.?|msnm|\bm\b)", re.I)


def _celda(t):
    t = _html.unescape(_html.unescape(str(t or "")))
    t = re.sub(r"<br\s*/?>", " ", t, flags=re.I)
    return re.sub(r"\s+", " ", t).strip(" :*")


def leer_tabla_md(md):
    filas = []
    for ln in (md or "").splitlines():
        ln = ln.strip()
        if not ln.startswith("|") or re.fullmatch(r"\|?\s*:?-{2,}.*", ln) or set(ln) <= set("|-: "):
            continue
        filas.append([_celda(c) for c in ln.strip("|").split("|")])
    return filas


def _unidad(txt, pred):
    m = UNIDAD.search(txt or "")
    if m:
        u = re.sub(r"\s+", "", m.group(1)).replace("²", "2").replace("³", "3")
        return "m s.n.m." if "s" in u.lower() and "n" in u.lower() else u
    return "m" if pred in ("tieneAltitud", "tieneProfundidadMax") else ""


def _num_celda(txt):
    m = re.search(r"-?\d{1,3}(?:[ .]\d{3})+(?:,\d+)?|-?\d+(?:[.,]\d+)?", txt or "")
    return (numero(m.group(0)), m.group(0)) if m else (None, "")


def tripletas_tablas(tablas, titulos=None):
    """Hechos desde tablas: (a) fichas clave-valor de una laguna; (b) inventarios con una fila por laguna/glaciar."""
    titulos = titulos or {}
    trip, menc = [], []
    for t in tablas:
        filas = leer_tabla_md(t.get("markdown", ""))
        if len(filas) < 2:
            continue
        base = {"paper_id": t["paper_id"], "chunk_id": t.get("table_id", ""), "section": "tabla",
                "page_start": t.get("page"), "page_end": t.get("page"), "source": t.get("fuente", "")}

        def add(s, p, o, tipo_o, valor=None, unidad="", ev=""):
            trip.append({**base, "subject": s["nombre"], "subject_type": s["tipo"], "predicate": p, "object": o,
                         "object_type": tipo_o, "value": valor, "unit": unidad, "year": "",
                         "evidence": f"Tabla (p. {t.get('page')}): {ev}"[:600], "confidence": 0.8, "method": "tabla"})

        ancho = max(len(f) for f in filas)
        # tabla "Parámetro | Valor | Unidad" → ficha clave-valor con la unidad en la 3.ª columna
        if ancho == 3 and re.search(r"par[aá]metro|caracter[ií]stica|descripci", filas[0][0], re.I) and re.search(r"valor", filas[0][1], re.I):
            filas = [[f[0], f"{f[1]} {f[2]}".strip()] for f in filas[1:] if len(f) == 3]
            ancho = 2
        # (a) ficha clave-valor: 2 columnas, claves conocidas
        if ancho == 2 and sum(bool(COL_NOMBRE.search(f[0]) or any(p.search(f[0]) for _, p in COL_MEDIDA)
                                   or any(p.search(f[0]) for _, p in COL_LUGAR)) for f in filas if len(f) == 2) >= 2:
            sujeto = None
            for f in filas:
                if len(f) == 2 and re.search(r"^(nombre|laguna)\b", f[0], re.I):
                    n = canon_nombre("Laguna", f[1])
                    sujeto = {"tipo": "Laguna", "nombre": n} if n else None
            if not sujeto:
                lags = {m["nombre"] for m in menciones_reglas(titulos.get(t["paper_id"], "")) if m["tipo"] == "Laguna"}
                sujeto = {"tipo": "Laguna", "nombre": lags.pop()} if len(lags) == 1 else None
            if not sujeto:
                continue
            menc.append({**base, "tipo": "Laguna", "nombre": sujeto["nombre"], "metodo": "tabla", "texto": sujeto["nombre"]})
            for f in filas:
                if len(f) != 2:
                    continue
                k, v = f
                for pred, pat in COL_MEDIDA:
                    if pat.search(k):
                        num, txt = _num_celda(v)
                        if num is not None:
                            u = _unidad(v, pred) or _unidad(k, pred)
                            add(sujeto, pred, f"{txt} {u}".strip(), "Literal", num, u, f"{k}: {v}")
                        break
                else:
                    for tipo_l, pat in COL_LUGAR:
                        if pat.search(k):
                            n = canon_nombre(tipo_l, v)
                            if n and not re.search(r"\d", n):
                                add(sujeto, "ubicadaEn", n, tipo_l, ev=f"{k}: {v}")
                                menc.append({**base, "tipo": tipo_l, "nombre": n, "metodo": "tabla", "texto": v[:80]})
                            break
            continue
        # (b) inventario: encabezado con columna de nombre + columnas de medidas o lugares
        enc = filas[0]
        i_nom = next((i for i, h in enumerate(enc) if COL_NOMBRE.search(h)), None)
        if i_nom is None or len(filas) < 3:
            continue
        tipo_s = ("Glaciar" if re.search(r"glaciar|glacier", enc[i_nom], re.I) else
                  "Nevado" if re.search(r"nevado", enc[i_nom], re.I) else
                  "Bofedal" if re.search(r"bofedal", enc[i_nom], re.I) else "Laguna")
        cols_med = [(i, pred, _unidad(h, pred)) for i, h in enumerate(enc) if i != i_nom
                    for pred, pat in COL_MEDIDA[:1] + COL_MEDIDA[1:] if pat.search(h)]
        cols_med = list({i: (i, p, u) for i, p, u in cols_med}.values())
        cols_lug = [(i, tl) for i, h in enumerate(enc) if i != i_nom for tl, pat in COL_LUGAR if pat.search(h)]
        if not cols_med and not cols_lug:
            continue
        for f in filas[1:]:
            if len(f) <= i_nom:
                continue
            n = canon_nombre(tipo_s, f[i_nom])
            if not n or re.fullmatch(r"[\d\s.,-]+", n) or re.search(r"^total|^sub ?total|^promedio", n, re.I):
                continue
            s = {"tipo": tipo_s, "nombre": n}
            menc.append({**base, "tipo": tipo_s, "nombre": n, "metodo": "tabla", "texto": f[i_nom][:80]})
            fila_txt = " | ".join(f)
            for i, pred, u in cols_med:
                if i < len(f):
                    num, txt = _num_celda(f[i])
                    if num is not None and (u or pred in ("tieneAltitud", "tieneProfundidadMax")):
                        if pred == "tieneVolumen" and tipo_s != "Laguna":
                            continue
                        add(s, pred, f"{txt} {u}".strip(), "Literal", num, u, fila_txt)
            for i, tl in cols_lug:
                if i < len(f):
                    lug = canon_nombre(tl, f[i])
                    if lug and not re.search(r"\d", lug):
                        add(s, "ubicadaEn", lug, tl, ev=fila_txt)
    return menc, trip


# ======================================================================
# 12. v2 — Diccionario de nombres construido con el corpus + búsqueda de nombres sueltos
# ======================================================================
def construir_diccionario(menciones, min_docs=2, min_largo=5):
    """Nombres de lagunas, glaciares, nevados y bofedales vistos en ≥ min_docs documentos.
    Excluye nombres que también son lugares administrativos o poblados (ambiguos)."""
    m = pd.DataFrame(menciones)
    if not len(m):
        return {}
    otros = set(m[m.tipo.isin(["Departamento", "Provincia", "Distrito", "CentroPoblado", "Cordillera", "Rio",
                               "Cuenca", "Quebrada", "Institucion"])].nombre.map(norm))
    cand = m[m.tipo.isin(["Laguna", "Glaciar", "Nevado", "Bofedal"])]
    g = cand.groupby(["nombre", "tipo"]).paper_id.nunique().reset_index(name="n_docs")
    g = g[(g.n_docs >= min_docs) & (g.nombre.str.len() >= min_largo) & ~g.nombre.map(norm).isin(otros)
          & ~g.nombre.str.fullmatch(r"[\d\s]+") & ~g.nombre.map(lambda x: norm(x) in NO_NOMBRES)]
    dicc = defaultdict(set)
    for r in g.itertuples():
        dicc[r.nombre].add(r.tipo)
    return dict(dicc)


def compilar_diccionario(dicc):
    if not dicc:
        return None
    nombres = sorted(dicc, key=len, reverse=True)
    return re.compile(r"(?<![\wÁÉÍÓÚÑáéíóúñü])(" + "|".join(re.escape(n) for n in nombres) + r")(?![\wÁÉÍÓÚÑáéíóúñü])")


def menciones_diccionario(texto, dicc, patron):
    """Nombres conocidos que aparecen sueltos. Si un nombre es laguna y glaciar a la vez (p. ej. Shallap),
    decide por la palabra anterior; si no hay pista, no lo usa."""
    out = []
    if not patron:
        return out
    for m in patron.finditer(texto):
        n = m.group(1)
        tipos = dicc.get(n, set())
        antes = texto[max(0, m.start() - 30):m.start()].lower()
        if len(tipos) == 1:
            tipo = next(iter(tipos))
        elif re.search(r"laguna|lake", antes):
            tipo = "Laguna"
        elif re.search(r"glaciar|glacier", antes):
            tipo = "Glaciar"
        elif re.search(r"nevado|mount", antes):
            tipo = "Nevado"
        else:
            continue
        if tipo in tipos:
            out.append({"tipo": tipo, "nombre": n, "ini": m.start(), "fin": m.end(), "metodo": "diccionario"})
    return out


# ======================================================================
# 13. v2 — Unificar nombres (variantes, errores de OCR)
# ======================================================================
def _clave_suelta(n):
    k = re.sub(r"[^a-z0-9 ]", "", norm(n))
    return re.sub(r"(.)\1+", r"\1", k.replace(" ", ""))   # "Palcaccocha" → "palcacocha"


SUF_DISTINTIVO = re.compile(r"\b(alta|baja|grande|chico|chica|nueva|vieja|norte|sur|este|oeste|superior|inferior|i{1,3}|iv|\d+)\b", re.I)


def resolver_nombres(menciones, umbral=93):
    """Mapa {(tipo, nombre) → nombre canónico}: une variantes del mismo tipo (misma clave sin tildes ni letras repetidas,
    o muy parecidas). No une nombres que se distinguen por un sufijo (Safuna Alta ≠ Safuna Baja)."""
    try:
        from rapidfuzz import fuzz
        parecido = lambda a, b: fuzz.ratio(a, b)
    except ImportError:
        from difflib import SequenceMatcher
        parecido = lambda a, b: 100 * SequenceMatcher(None, a, b).ratio()
    m = pd.DataFrame(menciones)
    alias = {}
    if not len(m):
        return alias
    frec = m.groupby(["tipo", "nombre"]).paper_id.nunique().reset_index(name="n")
    for tipo, g in frec.groupby("tipo"):
        if tipo in ("MetodoMonitoreo", "ObraSeguridad", "Ecosistema", "Fenomeno", "Especie", "ServicioEcosistemico", "Evento"):
            continue
        g = g.assign(tilde=g.nombre.map(lambda x: x != sin_tildes(x)), largo=g.nombre.str.len())
        g = g.sort_values(["n", "tilde", "largo"], ascending=False)   # en empate gana la forma con tildes
        canon = []          # (clave, nombre, sufijos)
        for r in g.itertuples():
            k = _clave_suelta(r.nombre)
            suf = set(x.lower() for x in SUF_DISTINTIVO.findall(r.nombre))
            destino = None
            for kc, nc, sc in canon:
                if suf != sc:
                    continue
                if k == kc or (len(k) >= 6 and k[:3] == kc[:3] and parecido(k, kc) >= umbral):
                    destino = nc
                    break
            if destino and destino != r.nombre:
                alias[(tipo, r.nombre)] = destino
            elif not destino:
                canon.append((k, r.nombre, suf))
    return alias


def aplicar_alias(registros, alias, campos=(("tipo", "nombre"),)):
    for x in registros:
        for ct, cn in campos:
            k = (x.get(ct), x.get(cn))
            if k in alias:
                x[cn] = alias[k]
    return registros


# ======================================================================
# 14. v2 — spaCy: sujeto → verbo → objeto entre entidades ya reconocidas
# ======================================================================
SUJ = {"nsubj", "nsubj:pass", "csubj"}
OBJ = {"obj", "obl", "iobj", "obl:agent", "obl:arg", "xcomp"}
SUBIR = {"appos", "flat", "flat:name", "compound", "nmod", "nummod", "amod"}
VERBOS_GENERICOS = {"marcar", "ver", "considerar", "realizar", "desarrollar", "observar", "mostrar", "indicar", "señalar",
                    "llamar", "denominar", "conocer", "utilizar", "usar", "tomar", "obtener", "permitir", "incluir",
                    "corresponder", "representar", "describir", "mencionar", "citar", "elaborar", "aplicar"}
VERBOS_LIGEROS = {"ser", "estar", "haber", "hacer", "poder", "deber", "tener", "ir", "dar", "parecer", "quedar"}
VERBO_PRED = {
    "amenazar": "amenazaA", "afectar": "afectoA", "destruir": "afectoA", "arrasar": "afectoA", "sepultar": "afectoA",
    "inundar": "afectoA", "devastar": "afectoA", "dañar": "afectoA", "alimentar": "alimentadaPor",
    "abastecer": "abasteceA", "proveer": "abasteceA", "suministrar": "abasteceA", "irrigar": "abasteceA",
    "regar": "abasteceA", "drenar": "drenaHacia", "desaguar": "drenaHacia", "desembocar": "drenaHacia",
    "verter": "drenaHacia", "descargar": "drenaHacia", "monitorear": "intervieneEn", "evaluar": "intervieneEn",
    "inspeccionar": "intervieneEn", "vigilar": "intervieneEn", "estudiar": "intervieneEn", "construir": "intervieneEn",
    "ejecutar": "intervieneEn", "ubicar": "ubicadaEn", "situar": "ubicadaEn", "localizar": "ubicadaEn",
    "encontrar": "ubicadaEn", "pertenecer": "ubicadaEn",
}
INVERTIR = {"alimentar"}   # "el glaciar X alimenta la laguna Y" → Y alimentadaPor X


def cargar_spacy(modelo="es_core_news_md"):
    import spacy
    try:
        return spacy.load(modelo, disable=["ner"])
    except OSError:
        from spacy.cli import download
        download(modelo)
        return spacy.load(modelo, disable=["ner"])


def _gobierna(tok):
    pasos = 0
    while tok.dep_ in SUBIR and tok.head.i != tok.i and pasos < 4:
        tok, pasos = tok.head, pasos + 1
    return tok


def _infinitivo(lema):
    """'monitoreado' → 'monitorear' (los participios a veces quedan como adjetivo)."""
    if lema in VERBO_PRED or not re.search(r"(ad|id)[oa]s?$", lema):
        return lema
    raiz = re.sub(r"(ad|id)[oa]s?$", "", lema)
    for fin in (("ar",) if re.search(r"ad[oa]s?$", lema) else ("er", "ir")):
        if raiz + fin in VERBO_PRED:
            return raiz + fin
    return raiz + ("ar" if re.search(r"ad[oa]s?$", lema) else "ir")


def _prep(tok):
    return next((c.lemma_.lower() for c in tok.children if c.dep_ == "case"), "")


def relaciones_spacy(nlp, chunks, menciones_por_chunk, cache_ruta=None, lote=32, log=print):
    """Para cada oración con 2+ entidades: si una es sujeto y otra objeto del mismo verbo → tripleta.
    Verbos conocidos se convierten en predicados de la ontología; el resto queda como 'relacionadoCon' + verbo."""
    hechos = {}
    if cache_ruta and os.path.exists(cache_ruta):
        for r in leer_jsonl(cache_ruta):
            hechos[r["chunk_id"]] = r
    def firma(ms):
        return str(sum(hash((m["tipo"], m["nombre"], m["ini"])) % 1000003 for m in ms))
    obj = [c for c in chunks if len(menciones_por_chunk.get(c["chunk_id"], [])) >= 2]
    pend = [c for c in obj if hechos.get(c["chunk_id"], {}).get("firma") != firma(menciones_por_chunk[c["chunk_id"]])]
    log(f"spaCy: {len(obj)} fragmentos con 2+ entidades · {len(obj) - len(pend)} en caché · {len(pend)} por analizar")
    fh = open(cache_ruta, "a", encoding="utf-8") if cache_ruta else None
    t0 = time.time()
    try:
        textos = (c["text"] for c in pend)
        for k, (c, doc) in enumerate(zip(pend, nlp.pipe(textos, batch_size=lote)), start=1):
            ms = menciones_por_chunk[c["chunk_id"]]
            trip = []
            base = {"paper_id": c["paper_id"], "chunk_id": c["chunk_id"], "section": c.get("section_type", ""),
                    "page_start": c.get("page_start"), "page_end": c.get("page_end"), "source": c.get("fuente", "")}
            ents = []
            for m in ms:
                if m["tipo"] in ("MetodoMonitoreo", "ServicioEcosistemico"):
                    continue
                sp = doc.char_span(m["ini"], m["fin"], alignment_mode="expand")
                if sp is not None:
                    ents.append((m, _gobierna(sp.root), sp.sent))
            for i, (a, ga, sa) in enumerate(ents):
                for b, gb, sb in ents[i + 1:] + ents[:i]:
                    if sa.start != sb.start or a["nombre"] == b["nombre"] or ga.i == gb.i:
                        continue
                    v = ga.head
                    pasiva_ser = (ga.dep_ in SUJ and gb.head.i == v.i and gb.dep_ in ("obl", "obl:agent", "obj") and _prep(gb) == "por"
                                  and any(ch.lemma_ == "ser" and ch.dep_.startswith("aux") for ch in v.children))
                    activa = not pasiva_ser and ga.dep_ in SUJ and gb.dep_ in OBJ and gb.head.i == v.i and v.pos_ in ("VERB", "AUX")
                    pasiva = (ga.dep_ == "nsubj:pass" and gb.head.i == v.i and gb.dep_ in ("obl", "obl:agent")
                              and _prep(gb) == "por")
                    participio = (gb.dep_ in ("obl", "obl:agent") and _prep(gb) == "por" and gb.head.dep_ in ("amod", "acl")
                                  and _gobierna(gb.head.head).i == ga.i)
                    pasiva = pasiva or pasiva_ser
                    if not (activa or pasiva or participio):
                        continue
                    tv = gb.head if participio else v
                    verbo = _infinitivo(tv.lemma_.lower())
                    # descarta nombres propios mal etiquetados como verbo ("Huascarán" → "huascacer") y verbos sin contenido
                    if (verbo in VERBOS_LIGEROS or verbo in VERBOS_GENERICOS or tv.text[:1].isupper()
                            or not re.fullmatch(r"[a-záéíóúñ]+(ar|er|ir|ír)", verbo)):
                        continue
                    suj, obj_ = (b, a) if (pasiva or participio) else (a, b)
                    if verbo in INVERTIR:
                        suj, obj_ = obj_, suj
                    pred = VERBO_PRED.get(verbo, "relacionadoCon")
                    regla = DOMINIO.get(pred)
                    if regla and (suj["tipo"] not in regla[0] or obj_["tipo"] not in regla[1]):
                        pred = "relacionadoCon"
                    prep = _prep(gb)
                    trip.append({**base, "subject": suj["nombre"], "subject_type": suj["tipo"], "predicate": pred,
                                 "object": obj_["nombre"], "object_type": obj_["tipo"], "verb": f"{verbo}{(' ' + prep) if prep and not participio else ''}",
                                 "value": None, "unit": "", "year": "", "evidence": sa.text.strip()[:600],
                                 "confidence": 0.65 if pred != "relacionadoCon" else 0.5, "method": "spacy"})
            reg = {"chunk_id": c["chunk_id"], "firma": firma(ms), "tripletas": trip}
            hechos[c["chunk_id"]] = reg
            if fh:
                fh.write(json.dumps(reg, ensure_ascii=False, default=str) + "\n")
            if k % 1000 == 0 or k == len(pend):
                vel = k / max(time.time() - t0, 1e-6)
                log(f"   spaCy {k}/{len(pend)} · {vel:.0f} frag/s · faltan ~{(len(pend) - k) / max(vel, 1e-6) / 60:.0f} min")
                if fh:
                    fh.flush()
    finally:
        if fh:
            fh.close()
    ids = {c["chunk_id"] for c in obj}
    return [t for cid, r in hechos.items() if cid in ids for t in r["tripletas"]]


# ======================================================================
# 15. v2 — LLM con varios proveedores (Gemini → Groq u otro compatible con OpenAI)
# ======================================================================
class ClienteOpenAICompat:
    """Cualquier API con formato OpenAI (Groq, Cerebras, GitHub Models…). Elige un modelo vigente de la lista del proveedor."""
    def __init__(self, api_key, base_url="https://api.groq.com/openai/v1", preferidos=(), pausa=2.5, nombre="groq"):
        import requests
        self.s, self.base, self.nombre, self.pausa = requests.Session(), base_url.rstrip("/"), nombre, pausa
        self.s.headers.update({"Authorization": f"Bearer {api_key}"})
        self._ultimo, self.json_mode = 0.0, True
        try:
            ids = [m["id"] for m in self.s.get(f"{self.base}/models", timeout=30).json().get("data", [])]
        except Exception:
            ids = []
        ids = [i for i in ids if not re.search(r"whisper|tts|guard|embed|vision|audio|speech|playai|compound|prompt", i, re.I)]
        buenos = [i for i in ids if re.search(r"70b|120b|maverick|qwen|versatile|large", i, re.I)]
        self.modelos = list(dict.fromkeys([p for p in preferidos if p in ids] + buenos + ids)) or list(preferidos) or ["llama-3.3-70b-versatile"]
        self.i_modelo = 0

    @property
    def modelo(self):
        return f"{self.nombre}:{self.modelos[self.i_modelo]}"

    def generar(self, prompt):
        espera = self.pausa - (time.time() - self._ultimo)
        if espera > 0:
            time.sleep(espera)
        cuerpo = {"model": self.modelos[self.i_modelo], "temperature": 0,
                  "messages": [{"role": "user", "content": prompt}]}
        if self.json_mode:
            cuerpo["response_format"] = {"type": "json_object"}
        self._ultimo = time.time()
        r = self.s.post(f"{self.base}/chat/completions", json=cuerpo, timeout=120)
        if r.status_code == 400 and "response_format" in r.text and self.json_mode:
            self.json_mode = False
            return self.generar(prompt)
        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code} {r.text[:300]}")
        return r.json()["choices"][0]["message"]["content"] or ""


def extraer_llm_varios(clientes, chunks, cache_ruta, log=print):
    """Usa el primer proveedor; si se satura o se agota, sigue con el siguiente sobre lo que falte."""
    hechos = {}
    for cli in clientes:
        hechos = extraer_llm(cli, chunks, cache_ruta, log=log)
        faltan = sum(c["chunk_id"] not in hechos for c in chunks)
        if not faltan:
            break
        log(f"   quedan {faltan} fragmentos → se intenta con el siguiente proveedor")
    return hechos


# ======================================================================
# 16. v2 — Redes de seguridad para preguntas no previstas + cobertura de preguntas
# ======================================================================
def co_menciones(df_menc, a, b, n=10):
    """Red 2: documentos (y fragmentos) que mencionan a la vez dos entidades, aunque no haya relación extraída."""
    ka, kb = norm(a), norm(b)
    m = df_menc.assign(k=df_menc.nombre.map(norm))
    ca = set(m[m.k == ka].chunk_id)
    cb = set(m[m.k == kb].chunk_id)
    juntos = ca & cb
    docs = set(m[m.k == ka].paper_id) & set(m[m.k == kb].paper_id)
    return {"fragmentos": sorted(juntos)[:n], "documentos": sorted(docs)[:n], "n_fragmentos": len(juntos), "n_documentos": len(docs)}


def buscar_texto(chunks, *terminos, n=8):
    """Red 3: fragmentos que contienen todos los términos (sin tildes ni mayúsculas), con su procedencia."""
    ks = [norm(t) for t in terminos]
    res = [c for c in chunks if all(k in norm(c["text"]) for k in ks)]
    res.sort(key=lambda c: -c.get("prioridad", 0))
    return [{"paper_id": c["paper_id"], "chunk_id": c["chunk_id"], "page_start": c.get("page_start"),
             "title": c.get("title", ""), "texto": c["text"][:400]} for c in res[:n]]


PREGUNTAS_COMPETENCIA = {
    "P1 Lagunas formadas en la Cordillera Blanca": ["formadaEnAnio", "ubicadaEn"],
    "P2 Lagunas con riesgo muy alto en Áncash": ["tieneNivelRiesgo", "ubicadaEn"],
    "P3 Centros poblados aguas abajo y estudios": ["amenazaA", "mencionaA"],
    "P4 Eventos de desborde → glaciar → poblados": ["ocurrioEn", "alimentadaPor", "afectoA"],
    "P5 Glaciares con balance de masa → lagunas en riesgo": ["monitoreadoCon", "alimentadaPor", "tieneNivelRiesgo"],
    "P6 Métodos de monitoreo e institución": ["monitoreadoCon", "intervieneEn", "usaMetodo"],
    "P7 Subcuencas con riesgo + drenaje ácido de roca": ["presentaFenomeno", "tieneNivelRiesgo", "ubicadaEn"],
    "P8 Bofedales en subcuencas con retroceso + servicios": ["tieneEcosistema", "presentaFenomeno", "proveeServicio"],
    "P9 Superficie glaciar 2018 vs 2023": ["tieneSuperficieGlaciar"],
    "P10 Lagunas con obras de seguridad, estado y poblados": ["tieneObraSeguridad", "estadoObra", "amenazaA"],
}


def cobertura_preguntas(cons):
    filas = []
    for p, preds in PREGUNTAS_COMPETENCIA.items():
        cuentas = {x: int((cons.predicate == x).sum()) for x in preds}
        estado = "✅" if all(v >= 5 for v in cuentas.values()) else ("🟡" if any(v > 0 for v in cuentas.values()) else "❌")
        filas.append({"pregunta": p, "estado": estado, **{"hechos": ", ".join(f"{k}: {v}" for k, v in cuentas.items())}})
    return pd.DataFrame(filas)


import pandas as pd
from IPython.display import display, Image
papers = pd.read_csv(os.path.join(CORPUS, "papers.csv"), dtype={"paper_id": str})
docs_p2 = pd.read_csv(os.path.join(TEXTO, "documentos_procesados.csv"), dtype={"paper_id": str})
_r = os.path.join(TEXTO, "chunks.jsonl")
chunks = leer_jsonl(_r if os.path.exists(_r) else _r + ".gz")
cons = pd.read_csv(os.path.join(TRIP, "tripletas_consolidadas.csv"), dtype={"paper_id": str}, low_memory=False)
entidades = pd.read_csv(os.path.join(TRIP, "entidades.csv"))
pd.set_option("display.max_colwidth", 70)
print(f"Listo ✅ · {len(papers)} documentos · {len(chunks):,} fragmentos · {len(cons):,} hechos · datos de {os.path.basename(TRIP)}")

## 1 · Un PDF en vivo → texto estructurado

In [ ]:
t0 = time.time()
ruta_pdf = os.path.join(CORPUS, "pdfs", nombre_archivo(DOC) + ".pdf")
fila = papers.set_index("paper_id").loc[DOC]
res = procesar_pdf(ruta_pdf, DOC, fila.title, fila.abstract, extraer_tablas=True, max_paginas_tablas=10)
print(f"📄 {fila.title}\n   {fila.fuente} · {res['n_pages']} páginas · índice quitado (pág. {res['paginas_indice']}) · "
      f"{res['lineas_cabecera_pie_quitadas']} cabeceras/pies quitados · {time.time() - t0:.1f} s\n")
for s in res["sections"]:
    if s["n_chars"] > 0 and s["type"] not in ("anexos",):
        print(f"   [{s['type']:15s}] {s['heading'][:45]:45s} p.{s['page_start']}-{s['page_end']}")
if res["tables"]:
    print("\nTabla encontrada (p. %d):\n%s" % (res["tables"][0]["page"], res["tables"][0]["markdown"][:400]))

## 2 · Del texto a tripletas con su fuente

In [ ]:
frag = fragmentar(res, {"fuente": fila.fuente, "title": fila.title, "year": fila.year})
trip = [t for c in frag for t in extraer_chunk(c)[1]]
menc_t, trip_t = tripletas_tablas([{**t, "paper_id": DOC, "table_id": f"{DOC}::t{k}"} for k, t in enumerate(res["tables"])],
                                  {DOC: fila.title})
df = preparar_tripletas(trip + trip_t)
df, _ = validar_tripletas(df)
vista = df[df.predicate.isin(["ubicadaEn", "alimentadaPor", "tieneArea", "tieneAltitud", "tieneObraSeguridad",
                              "monitoreadoCon", "tieneNivelRiesgo", "amenazaA", "presentaFenomeno"])]
vista = vista.drop_duplicates(["subject", "predicate", "object"])
print(f"{len(frag)} fragmentos → {len(df)} tripletas · {time.time() - t0:.1f} s en total\n")
display(vista[["subject", "predicate", "object", "page_start", "evidence"]].head(12).reset_index(drop=True))

## 3 · El corpus completo

In [ ]:
rel = cons[~cons.predicate.isin(["mencionaA", "usaMetodo"])]
resumen = pd.DataFrame([
    ("Documentos (INAIGEM · ANA · SENAMHI)", f"{len(papers)}  ({', '.join(f'{k} {v}' for k, v in papers.fuente.value_counts().items())})"),
    ("Páginas procesadas (con OCR en escaneados)", f"{int(docs_p2.n_pages.fillna(0).sum()):,}"),
    ("Fragmentos con procedencia", f"{len(chunks):,}"),
    ("Entidades distintas", f"{len(entidades):,}"),
    ("Hechos (tripletas únicas)", f"{len(rel):,}"),
    ("Confirmados por 2+ documentos", f"{int((rel.n_docs >= 2).sum()):,}"),
] + ([("Confirmados por 2+ técnicas", f"{int((rel.n_metodos >= 2).sum()):,}")] if "n_metodos" in rel else []),
    columns=["Indicador", "Valor"])
display(resumen)
print("Lagunas más estudiadas:", ", ".join(f"{r.nombre} ({r.n_docs})" for r in entidades[entidades.tipo == "Laguna"].sort_values("n_docs", ascending=False).head(6).itertuples()))

## 4 · Preguntas de competencia (varios saltos, con cita)

In [ ]:
print("¿Qué lagunas tienen riesgo MUY ALTO en Áncash?   laguna → cordillera/provincia → departamento")
r = consulta_riesgo(cons, "Áncash", "Muy alto")
display(r[["laguna", "documento", "pagina", "evidencia"]].head(6).reset_index(drop=True))

print("\n¿Qué centros poblados amenaza o afectó Palcacocha?")
p = consulta_poblados_amenazados(cons, "Palcacocha")
display(p[["subject", "predicate", "object", "n_docs", "paper_id", "page_start"]].head(5).reset_index(drop=True))

## 5 · Así se verá el grafo de conocimiento

In [ ]:
import matplotlib.pyplot as plt
img = os.path.join(TRIP, "vecindario_Palcacocha.png")
if os.path.exists(img):
    display(Image(filename=img, width=900))
else:
    dibujar_vecindario(cons[cons.confianza >= 0.6], "Palcacocha")